# 🐍 Snake — an AI chess bot that trains live against Stockfish

**What this notebook builds**

| # | Component | Where |
|---|-----------|-------|
| 1 | Stockfish connection (teacher + sparring partner) | Part 1 |
| 2 | **Legal-move identifier**, board/move encoding, neural net (`snake_core.py`) | Part 2 |
| 3 | **Database** (SQLite) + the **learning module**: eval-drop punishment, post-game study sessions (`snake_learn.py`) | Part 3 |
| 4 | **Chess game identifier** (opening / phase / how it ended) — optional | Part 2 |
| 5 | **Real-time training**: Snake plays Stockfish and learns *during* the game | Part 4–5 |
| 6 | **Local website**: play Snake / Stockfish, **watch them play each other**, Snake's **prediction in %**, board editor & themes, **Snake studies every finished game** | Part 6 |
| 7 | **Public website** deploy bundle (Docker → Hugging Face Spaces / Render / etc.) | Part 7 |

**How Snake learns (the idea in 60 seconds)**

```
          ┌──────────────┐   plays vs    ┌─────────────────────┐
          │    Snake     │ ────────────▶ │ Stockfish (opponent, │
          │ (CNN policy  │               │  skill-limited)      │
          │  + value)    │               └─────────────────────┘
          └──────┬───────┘
                 │ every position Snake faces
                 ▼
   ┌───────────────────────────┐   "what would you play here, and how good is it?"
   │ Stockfish (teacher, full  │ ──────────────────────────────┐
   │  strength, 50 ms/position)│                               ▼
   └───────────────────────────┘                    ┌────────────────────┐
                                                    │ SQLite database    │
        gradient step(s) after every Snake move ◀── │ (replay buffer)    │
                                                    └────────────────────┘
```

* **Policy head** learns Stockfish's best move (imitation). **Value head** learns Stockfish's evaluation.
* Because Snake chooses its *own* moves in the games it learns from (not only Stockfish's), it learns to recover from **its own mistakes** — this is called DAgger-style on-policy imitation, and it works much better than training on Stockfish-vs-Stockfish games.
* **Punishing dumb moves:** for every move Snake chooses, Stockfish also scores the position *after* it. The **difference** `eval_after − eval_before` is what gets punished — so staying in an already-lost position costs nothing, but a move that throws away half a pawn or more does. (Details in Part 4.)
* **Two brains to choose from:** a ResNet CNN (default) or a **Transformer** (every square is a token; the policy is a from→to attention map). Switch with `cfg.arch`.
* A decaying "teacher forcing" probability occasionally plays Stockfish's move instead, so games reach deeper positions early on.
* The opponent's **Skill Level** rises automatically once Snake starts scoring.

**Honest expectations:** on a CPU, after an hour or two Snake will play sensible-looking chess and beat Stockfish at skill level 0–3 sometimes; it will *not* approach full Stockfish strength (that needs search + millions of positions). The pipeline is real though — leave it running longer (or on a GPU) and it keeps improving.

**Licensing note:** Stockfish (GPLv3) is used as a separate program that we talk to over the UCI protocol; we don't copy its code. `python-chess` is also GPL-3.0+. Hosting a website that uses them is fine; if you *distribute* this project (or the Docker image with Stockfish inside) you must keep it GPLv3-compatible and offer the source. The deploy bundle below doesn't contain Stockfish at all — only Snake's own weights.

## Part 1 — Install & connect Stockfish

In [ ]:
# Run once. (In Colab/Jupyter this works as-is.)
%pip install -q python-chess torch numpy flask matplotlib

**Install Stockfish** (the engine binary), then re-run the next cell:

* **Windows:** download from <https://stockfishchess.org/download/>, unzip, set `STOCKFISH_PATH` below to the `.exe`
* **macOS:** `brew install stockfish`
* **Linux / Colab:** `sudo apt-get install -y stockfish` (binary lands in `/usr/games/stockfish`)

In [ ]:
import os, shutil, time, math, random, sqlite3, threading, traceback
from collections import deque
from dataclasses import dataclass, field
import numpy as np
import chess, chess.engine, chess.svg
import torch, torch.nn.functional as F
import matplotlib.pyplot as plt
from IPython.display import SVG, display, clear_output

# ---- If auto-detect fails, put your path here, e.g. r"C:\stockfish\stockfish-windows-x86-64-avx2.exe"
MANUAL_STOCKFISH_PATH = None

def find_stockfish():
    candidates = [MANUAL_STOCKFISH_PATH, os.environ.get("STOCKFISH_PATH"), shutil.which("stockfish"),
                  "/usr/games/stockfish", "/usr/local/bin/stockfish", "/opt/homebrew/bin/stockfish"]
    for c in candidates:
        if c and os.path.exists(c):
            return c
    raise FileNotFoundError("Stockfish not found - install it and/or set MANUAL_STOCKFISH_PATH above.")

SF_PATH = find_stockfish()
with chess.engine.SimpleEngine.popen_uci(SF_PATH) as eng:
    print("Connected to:", eng.id["name"])
    print("Skill Level range:", eng.options["Skill Level"].min, "-", eng.options["Skill Level"].max)

## Part 2 — Core: legal-move identifier, network, agent, game identifier

This cell writes `snake_core.py` to disk (so the website and the deploy bundle reuse *exactly* the same code), then imports it.

Key ideas:
* **Legal-move identifier** — `legal_mask(board)` marks the 4096 possible *from→to* moves that are legal right now (pins, checks, castling, en-passant… all handled by `python-chess`). Snake's output is masked with it, so **it can never play an illegal move**.
* The board is always encoded **from the side-to-move's point of view**, so Snake learns one way of playing for both colours.
* `identify_game(board)` is the optional **game identifier**: opening name, phase, material and termination.

In [ ]:
%%writefile snake_core.py
"""
snake_core.py  -  the "brain" of Snake
  * board encoding + move encoding (side-to-move perspective)
  * legal-move identifier (masking)
  * SnakeNet (ResNet) and SnakeTransformer - two interchangeable policy+value networks (PyTorch)
  * SnakeAgent (move selection, save / load)
  * identify_game (opening / phase / termination detector)
"""
import math
import threading
import numpy as np
import chess
import torch
import torch.nn as nn
import torch.nn.functional as F

POLICY_SIZE = 64 * 64      # from-square * 64 + to-square (promotions -> queen)
N_PLANES = 18


# --------------------------------------------------------------------------
# 1. Move encoding + LEGAL MOVE IDENTIFIER
# --------------------------------------------------------------------------
# The board is always shown to the network from the point of view of the side
# to move (black positions are mirrored), so the net learns one "colour-free"
# way of playing. Moves are mirrored the same way.

def _orient(sq, turn):
    return sq if turn == chess.WHITE else chess.square_mirror(sq)


def move_to_index(move, turn):
    return _orient(move.from_square, turn) * 64 + _orient(move.to_square, turn)


def index_to_move(idx, board):
    f, t = divmod(int(idx), 64)
    f, t = _orient(f, board.turn), _orient(t, board.turn)   # mirror is its own inverse
    promo = None
    if board.piece_type_at(f) == chess.PAWN and chess.square_rank(t) in (0, 7):
        promo = chess.QUEEN                                  # auto-queen
    return chess.Move(f, t, promotion=promo)


def legal_mask(board):
    """Boolean vector of length 4096: True where the move is LEGAL right now."""
    mask = np.zeros(POLICY_SIZE, dtype=bool)
    for m in board.legal_moves:                              # python-chess knows every rule
        mask[move_to_index(m, board.turn)] = True            # (pins, checks, castling, e.p. ...)
    return mask


def is_legal_uci(board, uci):
    try:
        return chess.Move.from_uci(uci) in board.legal_moves
    except ValueError:
        return False


def encode_board(board):
    """18 x 8 x 8 float tensor: 12 piece planes, 4 castling, 1 en-passant, 1 halfmove clock."""
    b = board if board.turn == chess.WHITE else board.mirror()
    x = np.zeros((N_PLANES, 8, 8), dtype=np.float32)
    for sq, piece in b.piece_map().items():
        plane = piece.piece_type - 1 + (0 if piece.color == chess.WHITE else 6)
        x[plane, sq // 8, sq % 8] = 1.0
    x[12] = b.has_kingside_castling_rights(chess.WHITE)
    x[13] = b.has_queenside_castling_rights(chess.WHITE)
    x[14] = b.has_kingside_castling_rights(chess.BLACK)
    x[15] = b.has_queenside_castling_rights(chess.BLACK)
    if b.ep_square is not None:
        x[16, b.ep_square // 8, b.ep_square % 8] = 1.0
    x[17] = min(b.halfmove_clock, 100) / 100.0
    return x


# --------------------------------------------------------------------------
# 2. Neural network
# --------------------------------------------------------------------------
class ResBlock(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.c1 = nn.Conv2d(c, c, 3, padding=1, bias=False)
        self.b1 = nn.BatchNorm2d(c)
        self.c2 = nn.Conv2d(c, c, 3, padding=1, bias=False)
        self.b2 = nn.BatchNorm2d(c)

    def forward(self, x):
        h = F.relu(self.b1(self.c1(x)))
        h = self.b2(self.c2(h))
        return F.relu(x + h)


class SnakeNet(nn.Module):
    def __init__(self, channels=64, blocks=4):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(N_PLANES, channels, 3, padding=1, bias=False),
                                  nn.BatchNorm2d(channels), nn.ReLU())
        self.tower = nn.Sequential(*[ResBlock(channels) for _ in range(blocks)])
        self.policy = nn.Sequential(nn.Conv2d(channels, 32, 1, bias=False), nn.BatchNorm2d(32),
                                    nn.ReLU(), nn.Flatten(), nn.Linear(32 * 64, POLICY_SIZE))
        self.value = nn.Sequential(nn.Conv2d(channels, 8, 1, bias=False), nn.BatchNorm2d(8),
                                   nn.ReLU(), nn.Flatten(), nn.Linear(8 * 64, 64), nn.ReLU(),
                                   nn.Linear(64, 1), nn.Tanh())

    def forward(self, x):
        h = self.tower(self.stem(x))
        return self.policy(h), self.value(h).squeeze(-1)


class SnakeTransformer(nn.Module):
    """
    Transformer variant: every square is a token (64 tokens). Self-attention lets squares
    "talk" to each other. The policy is a from-square x to-square attention map (4096 logits),
    the value comes from the mean-pooled tokens.
    """
    def __init__(self, d_model=96, layers=4, heads=4):
        super().__init__()
        assert d_model % heads == 0, "channels (d_model) must be divisible by heads"
        self.d = d_model
        self.embed = nn.Linear(N_PLANES, d_model)
        self.pos = nn.Parameter(torch.randn(1, 64, d_model) * 0.02)
        layer = nn.TransformerEncoderLayer(d_model, heads, dim_feedforward=d_model * 4, dropout=0.0,
                                           batch_first=True, norm_first=True)
        self.enc = nn.TransformerEncoder(layer, layers, enable_nested_tensor=False)
        self.norm = nn.LayerNorm(d_model)
        self.q = nn.Linear(d_model, d_model)
        self.k = nn.Linear(d_model, d_model)
        self.value = nn.Sequential(nn.Linear(d_model, 64), nn.ReLU(), nn.Linear(64, 1), nn.Tanh())

    def forward(self, x):                                   # x: B x 18 x 8 x 8
        t = x.flatten(2).transpose(1, 2)                    # B x 64 x 18  (token index = square)
        h = self.norm(self.enc(self.embed(t) + self.pos))
        logits = (self.q(h) @ self.k(h).transpose(1, 2)) / math.sqrt(self.d)   # B x 64(from) x 64(to)
        return logits.flatten(1), self.value(h.mean(1)).squeeze(-1)


def build_net(arch="resnet", channels=64, blocks=4, heads=4):
    if arch == "transformer":
        return SnakeTransformer(channels, blocks, heads)
    if arch == "resnet":
        return SnakeNet(channels, blocks)
    raise ValueError(f"unknown arch {arch!r} (use 'resnet' or 'transformer')")


# --------------------------------------------------------------------------
# 3. Agent
# --------------------------------------------------------------------------
class SnakeAgent:
    def __init__(self, channels=64, blocks=4, device=None, arch="resnet", heads=4):
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.net = build_net(arch, channels, blocks, heads).to(self.device)
        self.lock = threading.RLock()        # training thread + web server share the net
        self.meta = {"arch": arch, "channels": channels, "blocks": blocks, "heads": heads,
                     "train_steps": 0, "games": 0, "sf_level": 0, "web_games": 0}

    @torch.no_grad()
    def evaluate(self, board):
        """-> (policy logits [4096] as numpy, value in [-1,1] for the side to move)"""
        x = torch.from_numpy(encode_board(board)).unsqueeze(0).to(self.device)
        with self.lock:
            self.net.eval()
            logits, value = self.net(x)
        return logits[0].cpu().numpy(), float(value[0])

    def select_move(self, board, temperature=0.0, lookahead=False, top_k=4, value_weight=1.0):
        """
        Pick a LEGAL move.
          temperature = 0   -> best move;  > 0 -> sample (exploration while training)
          lookahead   = True-> mate-in-1 check + re-rank the top_k moves with the value head
        """
        legal = list(board.legal_moves)
        if not legal:
            raise ValueError("no legal moves - game is over")
        logits, _ = self.evaluate(board)
        masked = np.where(legal_mask(board), logits, -1e9)      # illegal moves can never win

        if lookahead:
            for m in legal:                                      # always take a mate in 1
                board.push(m); mate = board.is_checkmate(); board.pop()
                if mate:
                    return m
            p = np.exp(masked - masked.max()); p /= p.sum()
            cand = np.argsort(-masked)[:min(top_k, len(legal))]
            best, best_score = None, -1e9
            for idx in cand:
                m = index_to_move(idx, board)
                board.push(m)
                if board.is_game_over():
                    child = 0.0                                  # draw-ish
                else:
                    child = -self.evaluate(board)[1]             # opponent's value, flipped
                board.pop()
                score = p[idx] + value_weight * child
                if score > best_score:
                    best, best_score = m, score
            return best

        if temperature <= 1e-6:
            idx = int(np.argmax(masked))
        else:
            z = (masked - masked.max()) / temperature
            p = np.exp(z); p /= p.sum()
            idx = int(np.random.choice(POLICY_SIZE, p=p))
        return index_to_move(idx, board)

    def predict(self, board, top=8):
        """Snake's 'thoughts': probability (in %) of every legal move + how well it thinks it is doing."""
        logits, v = self.evaluate(board)
        masked = np.where(legal_mask(board), logits, -1e9)
        p = np.exp(masked - masked.max()); p /= p.sum()
        rows = {}
        for m in board.legal_moves:
            if m.promotion and m.promotion != chess.QUEEN:
                continue                                         # under-promotions are not in the policy
            rows[m.uci()] = (float(p[move_to_index(m, board.turn)]) * 100.0, board.san(m))
        ranked = sorted(rows.items(), key=lambda kv: -kv[1][0])
        return {"value": v, "win_pct": (v + 1.0) * 50.0,        # expected score of the side to move
                "turn": "white" if board.turn else "black",
                "moves": [{"uci": u, "san": s, "pct": round(pc, 2)} for u, (pc, s) in ranked[:top]],
                "all": {u: round(pc, 3) for u, (pc, _) in ranked}}

    # ---- persistence ----
    def save(self, path):
        with self.lock:
            torch.save({"state_dict": self.net.state_dict(), "meta": self.meta}, path)

    @classmethod
    def load(cls, path, device=None):
        ck = torch.load(path, map_location="cpu", weights_only=False)
        meta = ck["meta"]
        meta.setdefault("arch", "resnet"); meta.setdefault("heads", 4); meta.setdefault("web_games", 0)
        agent = cls(meta["channels"], meta["blocks"], device,
                    arch=meta.get("arch", "resnet"), heads=meta.get("heads", 4))   # old files = resnet
        agent.net.load_state_dict(ck["state_dict"])
        agent.meta = meta
        agent.net.eval()
        return agent


# --------------------------------------------------------------------------
# 4. (Optional) Chess GAME IDENTIFIER: opening, phase, how the game ended
# --------------------------------------------------------------------------
OPENINGS = {
    "e4 e5 Nf3 Nc6 Bb5": "Ruy Lopez", "e4 e5 Nf3 Nc6 Bc4": "Italian Game",
    "e4 e5 Nf3 Nc6 d4": "Scotch Game", "e4 e5 Nf3 Nc6 Nc3 Nf6": "Four Knights Game",
    "e4 e5 Nf3 Nf6": "Petrov's Defence", "e4 e5 f4": "King's Gambit",
    "e4 e5 Nc3": "Vienna Game", "e4 e5 Bc4": "Bishop's Opening", "e4 e5": "Open Game",
    "e4 c5": "Sicilian Defence", "e4 c5 Nf3 d6": "Sicilian: Najdorf/Dragon family",
    "e4 c5 Nf3 Nc6": "Sicilian: Classical", "e4 c5 c3": "Sicilian: Alapin",
    "e4 e6": "French Defence", "e4 c6": "Caro-Kann Defence", "e4 d5": "Scandinavian Defence",
    "e4 d6": "Pirc Defence", "e4 g6": "Modern Defence", "e4 Nf6": "Alekhine's Defence",
    "d4 d5 c4 e6": "Queen's Gambit Declined", "d4 d5 c4 dxc4": "Queen's Gambit Accepted",
    "d4 d5 c4": "Queen's Gambit", "d4 d5 Bf4": "London System", "d4 d5": "Closed Game",
    "d4 Nf6 c4 g6": "King's Indian / Grunfeld setup", "d4 Nf6 c4 e6 Nc3 Bb4": "Nimzo-Indian Defence",
    "d4 Nf6 c4 e6": "Queen's Indian / Nimzo setup", "d4 Nf6 Bf4": "London System",
    "d4 Nf6": "Indian Defence", "d4 f5": "Dutch Defence",
    "c4": "English Opening", "Nf3": "Reti Opening", "f4": "Bird's Opening",
    "b3": "Nimzo-Larsen Attack", "g3": "King's Fianchetto Opening",
}
_OPENING_KEYS = sorted(((k.split(), v) for k, v in OPENINGS.items()), key=lambda kv: -len(kv[0]))


def identify_game(board):
    """Describe a game from its move stack: opening, phase, material, termination."""
    root = board.root()
    custom = root.fen() != chess.STARTING_FEN
    replay, sans = root.copy(), []
    for m in board.move_stack:
        sans.append(replay.san(m)); replay.push(m)
    sans_clean = [s.rstrip("+#") for s in sans]
    opening = "Unknown / out of book"
    for tokens, name in _OPENING_KEYS:
        if not custom and sans_clean[:len(tokens)] == tokens:
            opening = name
            break
    if custom:
        opening = "Custom position"
    elif not sans:
        opening = "Starting position"

    values = {chess.KNIGHT: 3, chess.BISHOP: 3, chess.ROOK: 5, chess.QUEEN: 9}
    def material(color):
        return sum(v * len(board.pieces(pt, color)) for pt, v in values.items())
    total = material(chess.WHITE) + material(chess.BLACK)
    if board.fullmove_number <= 10 and total >= 56:
        phase = "opening"
    elif total <= 24:
        phase = "endgame"
    else:
        phase = "middlegame"

    info = {"opening": opening, "phase": phase, "ply": board.ply(),
            "material_white": material(chess.WHITE), "material_black": material(chess.BLACK),
            "in_check": board.is_check(), "over": False, "result": "*", "termination": None,
            "winner": None}
    outcome = board.outcome(claim_draw=True)
    if outcome:
        info.update(over=True, result=outcome.result(), termination=outcome.termination.name,
                    winner=None if outcome.winner is None else ("white" if outcome.winner else "black"))
    return info

In [ ]:
import importlib, snake_core; importlib.reload(snake_core)
from snake_core import *

# quick demo of the legal-move identifier + game identifier
b = chess.Board()
for u in "e2e4 e7e5 g1f3 b8c6 f1b5 a7a6".split():
    b.push_uci(u)
print("Legal moves now:", b.legal_moves.count(), "| mask says:", legal_mask(b).sum())
print("Is e5e4 legal?", is_legal_uci(b, "e5e4"), "| Is b5c6 legal?", is_legal_uci(b, "b5c6"))
print(identify_game(b))

## Part 3 — Database + the learning module

`snake_learn.py` holds everything about *learning* (so the website can use it too):

* **`SnakeDB`** (SQLite) — `games`, `positions` (every position Snake faced: Stockfish's best move, eval **before**, eval **after Snake's move**, the eval change), `train_log`. Older `snake.db` files are **migrated automatically** — nothing you've collected is lost.
* **`Trainer`** — the loss function: imitate Stockfish + learn its evaluation + **punish eval-dropping moves** + reward near-best alternatives. Every weight/threshold lives in `LossCfg`.
* **`LearningHub`** — the "study session": after a game finishes, Stockfish labels it, then Snake re-reads it for several epochs (mixed with old memories) and the weights change.

In [ ]:
%%writefile snake_learn.py
"""
snake_learn.py - everything about LEARNING
  * LossCfg / LearnCfg : every knob, with UI metadata (the website builds its "Learning lab" from it)
  * SnakeDB            : SQLite memory (auto-migrates older snake.db files)
  * Trainer            : imitation + value + eval-DROP punishment + good-move reward
  * LearningHub        : "study session" after a game: label with Stockfish, then consolidate into the weights
"""
import math, os, queue, random, sqlite3, threading, time, traceback
from dataclasses import dataclass, field, fields, asdict
import numpy as np
import chess, chess.engine
import torch
import torch.nn.functional as F
from snake_core import encode_board, legal_mask, move_to_index, identify_game


# ==========================================================================
# 1. Configuration  (all of it is editable live from the website / notebook)
# ==========================================================================
@dataclass
class LossCfg:
    w_imitate: float = 1.0          # learn Stockfish's best move
    w_value: float = 1.0            # learn Stockfish's evaluation
    w_punish: float = 1.0           # punish moves that DROP the evaluation
    w_reward: float = 0.3           # reinforce alternative moves that keep the evaluation
    punish_mode: str = "unlikelihood"   # "unlikelihood" (stable) | "pg" (policy-gradient) | "off"
    squash: bool = False            # True: measure drops with tanh(cp/scale) -> hopeless positions punished less
    squash_scale: float = 400.0
    threshold_cp: float = 40.0      # ignore eval drops smaller than this (analysis noise)
    norm_cp: float = 300.0          # a drop of this many centipawns = full-strength punishment
    cp_clip: float = 1000.0         # clip evals to +-this before taking the difference (mate = 10000)
    reward_threshold_cp: float = 15.0   # a non-best move losing <= this is "as good as best"
    value_scale: float = 400.0      # value target = tanh(cp / value_scale)
    log_floor: float = -8.0         # PG mode: stop pushing log-prob down below this


@dataclass
class LearnCfg:
    auto_study: bool = True         # study every finished game on the website automatically
    learn_all_positions: bool = False   # also learn from the OTHER side's positions (imitate Stockfish)
    label_time: float = 0.10        # seconds Stockfish spends labelling each position
    epochs: int = 4                 # passes over the game
    batch_size: int = 16            # positions from the game per gradient step
    replay_ratio: float = 2.0       # extra old positions per game position (prevents forgetting)
    lr_scale: float = 1.0           # learning-rate multiplier during the study session
    save_after: bool = True         # write snake_model.pt after studying
    loss: LossCfg = field(default_factory=LossCfg)


# (label, help, min, max, step, options) for the UI. Anything missing falls back to a sensible default.
UI_META = {
    "auto_study": ("Study every finished game", "Snake learns from each game as soon as it ends"),
    "learn_all_positions": ("Learn from the other side too", "Also imitate Stockfish on positions where the opponent moved"),
    "label_time": ("Stockfish labelling time (s)", "More time = better labels, slower study", 0.02, 1.0, 0.01),
    "epochs": ("Study epochs", "How many times Snake re-reads the game", 1, 30, 1),
    "batch_size": ("Batch from the game", "Game positions per gradient step", 2, 128, 1),
    "replay_ratio": ("Replay ratio", "Old positions mixed in per game position", 0.0, 8.0, 0.5),
    "lr_scale": ("Learning-rate multiplier", "Study session speed (x base lr)", 0.1, 5.0, 0.1),
    "save_after": ("Save model after studying", "Write snake_model.pt"),
    "loss.w_imitate": ("Weight: imitate Stockfish", "", 0.0, 3.0, 0.05),
    "loss.w_value": ("Weight: value/eval", "", 0.0, 3.0, 0.05),
    "loss.w_punish": ("Weight: punishment", "Strength of the 'dumb move' penalty", 0.0, 5.0, 0.05),
    "loss.w_reward": ("Weight: reward", "Reinforce near-best alternative moves", 0.0, 3.0, 0.05),
    "loss.punish_mode": ("Punishment mode", "unlikelihood = -log(1-p) (stable); pg = policy gradient; off", None, None, None,
                         ["unlikelihood", "pg", "off"]),
    "loss.squash": ("Squash evals with tanh", "Hopeless positions: a further drop counts for less"),
    "loss.squash_scale": ("Squash scale (cp)", "", 100.0, 1000.0, 10.0),
    "loss.threshold_cp": ("Ignore drops below (cp)", "Dead-zone for analysis noise", 0.0, 300.0, 5.0),
    "loss.norm_cp": ("Full punishment at drop (cp)", "Punishment grows linearly up to this drop", 50.0, 1500.0, 10.0),
    "loss.cp_clip": ("Clip evals at (cp)", "Mate scores are clipped to this", 200.0, 3000.0, 50.0),
    "loss.reward_threshold_cp": ("Reward if loss under (cp)", "", 0.0, 100.0, 1.0),
}


def cfg_get(cfg, name):
    obj = cfg
    for part in name.split("."):
        obj = getattr(obj, part)
    return obj


def cfg_set(cfg, name, value):
    """Validated, clamped assignment. Returns the stored value."""
    *path, leaf = name.split(".")
    obj = cfg
    for part in path:
        obj = getattr(obj, part)
    current = getattr(obj, leaf)
    meta = UI_META.get(name, ())
    if isinstance(current, bool):
        value = bool(value)
    elif isinstance(current, int):
        value = int(round(float(value)))
    elif isinstance(current, float):
        value = float(value)
    else:
        value = str(value)
        opts = meta[5] if len(meta) > 5 else None
        if opts and value not in opts:
            raise ValueError(f"{name} must be one of {opts}")
    if isinstance(value, (int, float)) and not isinstance(value, bool) and len(meta) > 3 and meta[2] is not None:
        value = type(current)(min(max(value, meta[2]), meta[3]))
    setattr(obj, leaf, value)
    return value


def cfg_ui(cfg):
    """List of control descriptions for the website."""
    out = []
    for name, meta in UI_META.items():
        cur = cfg_get(cfg, name)
        item = {"name": name, "label": meta[0], "help": meta[1] if len(meta) > 1 else "", "value": cur}
        if isinstance(cur, bool):
            item["type"] = "bool"
        elif isinstance(cur, str):
            item.update(type="select", options=meta[5])
        else:
            item.update(type="int" if isinstance(cur, int) else "float", min=meta[2], max=meta[3], step=meta[4])
        out.append(item)
    return out


# ==========================================================================
# 2. Database
# ==========================================================================
SCHEMA = """
CREATE TABLE IF NOT EXISTS games(
  id INTEGER PRIMARY KEY AUTOINCREMENT, started_at REAL, snake_color TEXT, sf_level INTEGER,
  result TEXT, snake_score REAL, termination TEXT, opening TEXT, n_plies INTEGER, pgn TEXT);
CREATE TABLE IF NOT EXISTS positions(
  id INTEGER PRIMARY KEY AUTOINCREMENT, game_id INTEGER, ply INTEGER, fen TEXT,
  teacher_move TEXT, played_move TEXT, teacher_cp INTEGER, value_target REAL,
  phase TEXT, agreed INTEGER);
CREATE TABLE IF NOT EXISTS train_log(
  id INTEGER PRIMARY KEY AUTOINCREMENT, ts REAL, step INTEGER, policy_loss REAL,
  value_loss REAL, games INTEGER, sf_level INTEGER);
CREATE INDEX IF NOT EXISTS idx_pos_game ON positions(game_id);
"""
MIGRATIONS = {   # columns added later; older databases get them automatically
    "positions": [("cp_after", "INTEGER"), ("delta_cp", "INTEGER"), ("source", "TEXT")],
    "games": [("source", "TEXT")],
    "train_log": [("punish_loss", "REAL")],
}
ROW_COLS = "fen, teacher_move, value_target, played_move, teacher_cp, cp_after"


class SnakeDB:
    def __init__(self, path):
        self.path = path
        self.lock = threading.RLock()
        self.conn = sqlite3.connect(path, check_same_thread=False)
        self.conn.execute("PRAGMA journal_mode=WAL")
        self.conn.executescript(SCHEMA)
        for table, cols in MIGRATIONS.items():
            have = {r[1] for r in self.conn.execute(f"PRAGMA table_info({table})")}
            for name, typ in cols:
                if name not in have:
                    self.conn.execute(f"ALTER TABLE {table} ADD COLUMN {name} {typ}")
        self.conn.commit()

    def new_game(self, snake_color, sf_level, source="train"):
        with self.lock:
            cur = self.conn.execute("INSERT INTO games(started_at, snake_color, sf_level, source) VALUES(?,?,?,?)",
                                    (time.time(), snake_color, sf_level, source))
            self.conn.commit()
            return cur.lastrowid

    def add_position(self, game_id, ply, fen, teacher_move, played_move, cp, value_target, phase, agreed,
                     cp_after=None, delta_cp=None, source="train"):
        with self.lock:
            self.conn.execute(
                "INSERT INTO positions(game_id,ply,fen,teacher_move,played_move,teacher_cp,value_target,phase,"
                "agreed,cp_after,delta_cp,source) VALUES(?,?,?,?,?,?,?,?,?,?,?,?)",
                (game_id, ply, fen, teacher_move, played_move, cp, value_target, phase, int(agreed),
                 cp_after, delta_cp, source))
            self.conn.commit()

    def finish_game(self, game_id, result, snake_score, termination, opening, n_plies, pgn):
        with self.lock:
            self.conn.execute("UPDATE games SET result=?, snake_score=?, termination=?, opening=?, n_plies=?, pgn=? WHERE id=?",
                              (result, snake_score, termination, opening, n_plies, pgn, game_id))
            self.conn.commit()

    def log_train(self, step, p_loss, v_loss, games, sf_level, punish_loss=None):
        with self.lock:
            self.conn.execute("INSERT INTO train_log(ts,step,policy_loss,value_loss,games,sf_level,punish_loss) "
                              "VALUES(?,?,?,?,?,?,?)", (time.time(), step, p_loss, v_loss, games, sf_level, punish_loss))
            self.conn.commit()

    def count(self, table="positions"):
        with self.lock:
            return self.conn.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]

    def sample(self, n, recent_pool=500):
        """Half the batch from the newest positions (fresh mistakes), half uniformly from all history."""
        with self.lock:
            mx = self.conn.execute("SELECT MAX(id) FROM positions").fetchone()[0] or 0
            if mx == 0 or n <= 0:
                return []
            half = n // 2
            ids = [random.randint(max(1, mx - recent_pool + 1), mx) for _ in range(half)]
            ids += [random.randint(1, mx) for _ in range(n - half)]
            ids = list(set(ids))
            q = ",".join("?" * len(ids))
            return self.conn.execute(f"SELECT {ROW_COLS} FROM positions WHERE id IN ({q})", ids).fetchall()

    def query(self, sql, params=()):
        with self.lock:
            return self.conn.execute(sql, params).fetchall()


# ==========================================================================
# 3. Trainer  (imitation + value + punish eval-drops + reward)
# ==========================================================================
class Trainer:
    def __init__(self, agent, loss_cfg=None, lr=1e-3, weight_decay=1e-4):
        from collections import deque
        self.agent, self.loss_cfg, self.base_lr = agent, loss_cfg or LossCfg(), lr
        self.opt = torch.optim.AdamW(agent.net.parameters(), lr=lr, weight_decay=weight_decay)
        self.history = deque(maxlen=20000)     # (step, imitate_loss, value_loss, punish_loss)

    # ---- the key idea: use the DIFFERENCE between the eval before and after Snake's move ----
    def drop_strength(self, cp_before, cp_after):
        """0 .. 1 punishment strength for a move that took the eval from cp_before to cp_after
        (both from the mover's point of view). A move that merely stays in a lost position
        (before == after) has drop 0 -> not punished, however negative the eval is."""
        c = self.loss_cfg
        if cp_before is None or cp_after is None:
            return 0.0, 0.0
        b = max(-c.cp_clip, min(c.cp_clip, float(cp_before)))
        a = max(-c.cp_clip, min(c.cp_clip, float(cp_after)))
        if c.squash:
            s = c.squash_scale
            drop, thr, norm = math.tanh(b / s) - math.tanh(a / s), math.tanh(c.threshold_cp / s), math.tanh(c.norm_cp / s)
        else:
            drop, thr, norm = b - a, c.threshold_cp, c.norm_cp
        punish = min(max(drop - thr, 0.0) / max(norm, 1e-6), 1.0)
        reward = 1.0 if (b - a) <= c.reward_threshold_cp else 0.0      # (always judged in cp)
        return punish, reward

    def _prepare(self, rows):
        dev = self.agent.device
        boards = [chess.Board(r[0]) for r in rows]
        X = torch.from_numpy(np.stack([encode_board(b) for b in boards])).to(dev)
        M = torch.from_numpy(np.stack([legal_mask(b) for b in boards])).to(dev)
        y_best, y_mine, pun, rew, y_val = [], [], [], [], []
        for b, r in zip(boards, rows):
            _, tmove, _, pmove, cp_b, cp_a = r
            best = move_to_index(chess.Move.from_uci(tmove), b.turn)
            y_best.append(best)
            y_val.append(math.tanh(max(-10000, min(10000, cp_b if cp_b is not None else 0)) / self.loss_cfg.value_scale)
                         if cp_b is not None else float(r[2]))
            mine = move_to_index(chess.Move.from_uci(pmove), b.turn) if pmove else best
            y_mine.append(mine)
            if pmove and mine != best:
                p, rw = self.drop_strength(cp_b, cp_a)
                pun.append(p); rew.append(rw if cp_a is not None else 0.0)
            else:
                pun.append(0.0); rew.append(0.0)
        t = lambda v, dt=torch.float32: torch.tensor(v, dtype=dt, device=dev)
        return X, M, t(y_best, torch.long), t(y_mine, torch.long), t(pun), t(rew), t(y_val)

    def _loss(self, rows):
        c = self.loss_cfg
        X, M, y_best, y_mine, pun, rew, y_val = self._prepare(rows)
        logits, value = self.agent.net(X)
        logp = F.log_softmax(logits.masked_fill(~M, -1e9), dim=1)           # only LEGAL moves compete
        imitate = F.nll_loss(logp, y_best)
        v_loss = F.mse_loss(value, y_val)
        lp_mine = logp.gather(1, y_mine.unsqueeze(1)).squeeze(1)
        if c.punish_mode == "unlikelihood":
            pen = -torch.log1p(-lp_mine.exp().clamp(max=1 - 1e-4))          # -log(1 - p(move))
        elif c.punish_mode == "pg":
            pen = lp_mine.clamp(min=c.log_floor)                            # push log p(move) DOWN
        else:
            pen = torch.zeros_like(lp_mine)
        punish = (pun * pen).mean()
        reward = (rew * (-lp_mine)).mean()
        total = c.w_imitate * imitate + c.w_value * v_loss + c.w_punish * punish + c.w_reward * reward
        return total, imitate, v_loss, punish, reward, logp, y_best, int((pun > 0).sum())

    def step(self, rows, lr_scale=1.0):
        with self.agent.lock:
            for g in self.opt.param_groups:
                g["lr"] = self.base_lr * lr_scale
            self.agent.net.train()
            total, imitate, v_loss, punish, reward, _, _, n_pun = self._loss(rows)
            self.opt.zero_grad()
            total.backward()
            torch.nn.utils.clip_grad_norm_(self.agent.net.parameters(), 1.0)
            self.opt.step()
            self.agent.meta["train_steps"] += 1
            self.agent.net.eval()
            for g in self.opt.param_groups:
                g["lr"] = self.base_lr
        out = dict(imitate=imitate.item(), value=v_loss.item(), punish=punish.item(),
                   reward=reward.item(), n_punished=n_pun)
        self.history.append((self.agent.meta["train_steps"], out["imitate"], out["value"], out["punish"]))
        return out

    @torch.no_grad()
    def evaluate(self, rows):
        """No learning: how well does Snake currently match Stockfish on these positions?"""
        with self.agent.lock:
            self.agent.net.eval()
            _, imitate, v_loss, punish, _, logp, y_best, _ = self._loss(rows)
            agree = (logp.argmax(1) == y_best).float().mean().item()
        return dict(imitate=imitate.item(), value=v_loss.item(), punish=punish.item(), agree=agree)


# ==========================================================================
# 4. LearningHub: "study session" after a game
# ==========================================================================
class LearningHub:
    """
    After a game finishes (on the website), Snake 'sleeps on it':
      1) Stockfish labels every position Snake moved in (best move, eval before, eval after Snake's move)
      2) positions are stored in the database
      3) Snake re-reads the game for several epochs (mixed with old memories) -> weights change
    Jobs run one at a time in a background thread so the website stays responsive.
    """
    def __init__(self, agent, db, stockfish_path, cfg=None, trainer=None, model_path=None):
        self.agent, self.db, self.sf_path, self.model_path = agent, db, stockfish_path, model_path
        self.cfg = cfg or LearnCfg()
        self.trainer = trainer or Trainer(agent, self.cfg.loss)
        self.trainer.loss_cfg = self.cfg.loss                      # share live-editable loss settings
        self.q = queue.Queue()
        self.lock = threading.Lock()
        self.worker = None
        self.stop_event = threading.Event()
        self.jobs_done = 0
        self.status = {"state": "idle", "progress": 0, "total": 0, "message": "Snake is resting.",
                       "queue": 0, "last": None, "error": None}
        getattr(threading, "_register_atexit", lambda f: None)(self.stop_event.set)

    @property
    def enabled(self):
        return bool(self.sf_path and os.path.exists(self.sf_path))

    def get_status(self):
        with self.lock:
            s = dict(self.status); s["queue"] = self.q.qsize(); s["jobs_done"] = self.jobs_done
            return s

    def _set(self, **kw):
        with self.lock:
            self.status.update(kw)

    def submit(self, moves, start_fen=None, snake_color="white", source="web"):
        if not self.enabled:
            return {"queued": False, "reason": "Stockfish is not available, so Snake cannot label games."}
        if len(moves) < 2:
            return {"queued": False, "reason": "Game too short to study."}
        if snake_color not in ("white", "black", "none") or (snake_color == "none" and not self.cfg.learn_all_positions):
            return {"queued": False, "reason": "Snake did not play in this game (enable 'Learn from the other side too')."}
        self.q.put(dict(moves=list(moves), start_fen=start_fen, snake_color=snake_color, source=source))
        if self.worker is None or not self.worker.is_alive():
            self.worker = threading.Thread(target=self._loop, daemon=True)
            self.worker.start()
        return {"queued": True, "queue": self.q.qsize()}

    def _loop(self):
        while not self.stop_event.is_set():
            try:
                job = self.q.get(timeout=2.0)
            except queue.Empty:
                return                                  # idle: thread ends, restarts on next submit
            try:
                self._process(job)
            except Exception:
                self._set(state="error", error=traceback.format_exc(), message="Study session failed.")

    # ---------------------------------------------------------------
    def _terminal_cp(self, board, mover_color):
        if board.is_checkmate():
            return 10000 if board.turn != mover_color else -10000     # side to move is mated
        return 0

    def _label(self, job):
        cfg = self.cfg
        board = chess.Board(job["start_fen"]) if job["start_fen"] else chess.Board()
        snake = {"white": chess.WHITE, "black": chess.BLACK}.get(job["snake_color"])
        game_id = self.db.new_game(job["snake_color"], -1, source=job["source"])
        rows, limit = [], chess.engine.Limit(time=cfg.label_time)
        eng = chess.engine.SimpleEngine.popen_uci(self.sf_path)
        try:
            eng.configure({"Threads": 1, "Hash": 64, "Skill Level": 20})
            total = len(job["moves"])
            for i, uci in enumerate(job["moves"]):
                if self.stop_event.is_set():
                    break
                move = chess.Move.from_uci(uci)
                mine = snake is not None and board.turn == snake
                if mine or cfg.learn_all_positions:
                    info = eng.analyse(board, limit)
                    best = info["pv"][0]
                    cp_b = info["score"].pov(board.turn).score(mate_score=10000)
                    same = move_to_index(move, board.turn) == move_to_index(best, board.turn)
                    cp_a = cp_b
                    if not same:
                        after = board.copy(); after.push(move)
                        if after.is_game_over():
                            cp_a = self._terminal_cp(after, board.turn)
                        else:
                            cp_a = eng.analyse(after, limit)["score"].pov(board.turn).score(mate_score=10000)
                    delta = cp_a - cp_b if mine else None
                    self.db.add_position(game_id, board.ply(), board.fen(), best.uci(), uci if mine else None,
                                         cp_b, math.tanh(cp_b / self.cfg.loss.value_scale),
                                         identify_game(board)["phase"], same, cp_a if mine else None, delta,
                                         source=job["source"])
                    self.db.conn.commit()
                    rows.append((board.fen(), best.uci(), math.tanh(cp_b / self.cfg.loss.value_scale),
                                 uci if mine else None, cp_b, cp_a if mine else None))
                board.push(move)
                self._set(state="labelling", progress=i + 1, total=total,
                          message=f"Stockfish is labelling the game... move {i + 1}/{total}")
        finally:
            try:
                eng.quit()
            except Exception:
                pass
        return game_id, rows, board

    def _process(self, job):
        cfg, tr = self.cfg, self.trainer
        self._set(state="labelling", progress=0, total=len(job["moves"]), error=None,
                  message="Snake is going to sleep on this game...")
        game_id, rows, final = self._label(job)
        if not rows:
            self._set(state="idle", message="Nothing to learn from in that game."); return
        gi = identify_game(final)
        self.db.finish_game(game_id, gi["result"], None, gi["termination"] or "UNFINISHED", gi["opening"],
                            final.ply(), "")
        before = tr.evaluate(rows)
        snake_rows = [r for r in rows if r[3]]
        clip = lambda x: max(-cfg.loss.cp_clip, min(cfg.loss.cp_clip, x))     # one allowed mate must not dominate
        deltas = [clip(r[5]) - clip(r[4]) for r in snake_rows]
        n_pun = sum(1 for r in snake_rows if tr.drop_strength(r[4], r[5])[0] > 0)
        n_rew = sum(1 for r in snake_rows if r[3] and tr.drop_strength(r[4], r[5])[1] > 0
                    and chess.Move.from_uci(r[3]) != chess.Move.from_uci(r[1]))
        steps, total_steps = 0, cfg.epochs * max(1, math.ceil(len(rows) / cfg.batch_size))
        for ep in range(cfg.epochs):
            random.shuffle(rows)
            for k in range(0, len(rows), cfg.batch_size):
                if self.stop_event.is_set():
                    break
                chunk = rows[k:k + cfg.batch_size]
                replay = self.db.sample(int(len(chunk) * cfg.replay_ratio))
                tr.step(chunk + replay, lr_scale=cfg.lr_scale)
                steps += 1
                self._set(state="consolidating", progress=steps, total=total_steps,
                          message=f"Snake is embedding the game into memory... epoch {ep + 1}/{cfg.epochs}")
        after = tr.evaluate(rows)
        self.agent.meta["web_games"] = self.agent.meta.get("web_games", 0) + 1
        if cfg.save_after and self.model_path:
            self.agent.save(self.model_path)
        self.jobs_done += 1
        self._set(state="done", progress=total_steps, total=total_steps, message="Snake has learned from the game.",
                  last={"game_id": game_id, "positions": len(rows), "snake_moves": len(snake_rows),
                        "avg_delta_cp": round(float(np.mean(deltas)), 1) if deltas else None,
                        "punished": n_pun, "rewarded": n_rew, "steps": steps,
                        "imitate_before": round(before["imitate"], 3), "imitate_after": round(after["imitate"], 3),
                        "agree_before": round(before["agree"] * 100, 1), "agree_after": round(after["agree"] * 100, 1),
                        "train_steps_total": self.agent.meta["train_steps"]})

In [ ]:
import importlib, snake_learn; importlib.reload(snake_learn)
from snake_learn import *

db = SnakeDB("snake.db")
print("Database ready | positions:", db.count(), "| games:", db.count("games"))

## Part 4 — Real-time trainer & the Snake-vs-Stockfish loop

On every Snake move:
1. **Teacher** Stockfish (full strength) analyses the position → best move + eval **before** (`cp_before`).
2. Snake picks its own legal move. If it isn't Stockfish's move, the teacher scores the position **after** it (`cp_after`).
3. `delta = cp_after − cp_before` is stored in the database. A few gradient steps follow, on a batch from the database.
4. Stockfish (**opponent**, skill-limited) replies.

### How "dumb moves get punished" (and why we use the *difference*)
Both evals are from the mover's point of view. A move that merely stays in a lost position (−900 → −900) has `delta = 0` and is **not** punished — otherwise *every* move in a lost game would look terrible. A move that throws away value (0 → −400) has `delta = −400` and is punished in proportion:

`strength = clamp((drop − threshold) / norm, 0, 1)`

* `punish_mode="unlikelihood"` (default): loss `strength × −log(1 − p(move))` — lowers the probability Snake gives that move; bounded, so it's stable.
* `punish_mode="pg"`: policy-gradient style `strength × log p(move)`.
* `squash=True` measures drops through `tanh(cp/scale)`, so already-decided positions are punished even less.
* Moves that lose ≤ `reward_threshold_cp` but differ from Stockfish's choice are gently **reinforced** (`w_reward`).
* Tiny drops (< `threshold_cp`) are ignored: Stockfish at 50 ms is a little noisy.

All of this is in `cfg.loss` and can be changed **while training runs**. The dashboard's third plot (avg eval change per Snake move) should creep up towards 0 as Snake stops making dumb moves.

In [ ]:
@dataclass
class Cfg:
    stockfish_path: str = field(default_factory=lambda: SF_PATH)
    model_path: str = "snake_model.pt"
    # ---- the brain (only used when a NEW model is created) ----
    arch: str = "resnet"            # "resnet" or "transformer"
    channels: int = 64              # resnet: filters | transformer: d_model (96 is a good start)
    blocks: int = 4                 # resnet: residual blocks | transformer: encoder layers
    heads: int = 4                  # transformer attention heads (channels must divide by heads)
    # ---- Stockfish ----
    teacher_time: float = 0.05      # seconds per labelled position (full strength)
    opponent_time: float = 0.05     # seconds per opponent move
    start_level: int = 0            # opponent Skill Level (0-20)
    max_level: int = 20
    promote_score: float = 0.55     # avg score over last 10 games needed to level up
    # ---- exploration ----
    temperature: float = 0.7        # sampling temperature for Snake early in a game
    explore_plies: int = 16
    teacher_forcing_start: float = 0.5   # chance of playing Stockfish's move instead of Snake's
    teacher_forcing_decay: float = 0.97  # multiplied after each game
    teacher_forcing_min: float = 0.25
    # ---- learning ----
    lr: float = 1e-3
    weight_decay: float = 1e-3
    batch_size: int = 64
    steps_per_update: int = 1       # gradient steps after every Snake move
    min_buffer: int = 1000          # start learning once the DB has this many positions
    recent_pool: int = 500
    loss: LossCfg = field(default_factory=LossCfg)   # <- punishment / reward / weights (editable live)
    # ---- games ----
    max_plies: int = 200            # adjudicate as draw after this many half-moves
    save_every: int = 3             # checkpoint every N games


class TrainingSession:
    def __init__(self, agent, db, cfg):
        self.agent, self.db, self.cfg = agent, db, cfg
        self.trainer = Trainer(agent, cfg.loss, cfg.lr, cfg.weight_decay)
        self.level = max(cfg.start_level, agent.meta.get("sf_level", 0))
        self.games = agent.meta.get("games", 0)
        self.recent_scores = deque(maxlen=10)
        self.hist = []                          # per-game dicts for the dashboard
        self.board, self.status, self.error = chess.Board(), "idle", None
        self.stop_event, self.thread = threading.Event(), None
        self.current_agree = deque(maxlen=200)

    # ---- control ----
    @property
    def running(self):
        return self.thread is not None and self.thread.is_alive()

    def start(self, n_games=100):
        if self.running:
            print("Already running."); return
        self.stop_event.clear(); self.error = None
        self.thread = threading.Thread(target=self._run, args=(n_games,), daemon=True)
        self.thread.start()

    def stop(self):
        self.stop_event.set()
        if self.thread:
            self.thread.join(timeout=30)

    # ---- internals ----
    def _run(self, n_games):
        self.status = "running"
        teacher = opponent = None
        try:
            teacher = chess.engine.SimpleEngine.popen_uci(self.cfg.stockfish_path)
            teacher.configure({"Threads": 1, "Hash": 64})
            opponent = chess.engine.SimpleEngine.popen_uci(self.cfg.stockfish_path)
            opponent.configure({"Threads": 1, "Hash": 32, "Skill Level": self.level})
            for _ in range(n_games):
                if self.stop_event.is_set():
                    break
                self._play_game(teacher, opponent)
            self.status = "finished" if not self.stop_event.is_set() else "stopped"
        except Exception:
            self.error = traceback.format_exc(); self.status = "error"
        finally:
            for e in (teacher, opponent):
                try:
                    if e: e.quit()
                except Exception:
                    pass
            self.agent.save(self.cfg.model_path)

    def _learn(self):
        if self.db.count("positions") < self.cfg.min_buffer:
            return
        for _ in range(self.cfg.steps_per_update):
            rows = self.db.sample(self.cfg.batch_size, self.cfg.recent_pool)
            if len(rows) >= 8:
                out = self.trainer.step(rows)
                if self.agent.meta["train_steps"] % 50 == 0:
                    self.db.log_train(self.agent.meta["train_steps"], out["imitate"], out["value"],
                                      self.games, self.level, out["punish"])

    def _play_game(self, teacher, opponent):
        cfg = self.cfg
        limit = chess.engine.Limit(time=cfg.teacher_time)
        snake_color = chess.WHITE if self.games % 2 == 0 else chess.BLACK
        board = self.board = chess.Board()
        game_id = self.db.new_game("white" if snake_color else "black", self.level)
        beta = max(cfg.teacher_forcing_min, cfg.teacher_forcing_start * cfg.teacher_forcing_decay ** self.games)
        agree_game, delta_game = [], []
        self.status = "running"

        while not board.is_game_over(claim_draw=True) and board.ply() < cfg.max_plies:
            if self.stop_event.is_set():
                return
            if board.turn == snake_color:
                info = teacher.analyse(board, limit)
                best = info["pv"][0]
                cp_b = info["score"].pov(board.turn).score(mate_score=10000)            # eval BEFORE Snake's move
                temp = cfg.temperature if board.ply() < cfg.explore_plies else 0.0
                mine = self.agent.select_move(board, temperature=temp)
                same = move_to_index(mine, board.turn) == move_to_index(best, board.turn)
                cp_a = cp_b                                                              # best move keeps the eval
                if not same:                                                             # eval AFTER Snake's move
                    after = board.copy(); after.push(mine)
                    if after.is_game_over():
                        cp_a = 10000 if after.is_checkmate() else 0
                    else:
                        cp_a = teacher.analyse(after, limit)["score"].pov(snake_color).score(mate_score=10000)
                clip = lambda x: max(-cfg.loss.cp_clip, min(cfg.loss.cp_clip, x))
                delta_game.append(clip(cp_a) - clip(cp_b))
                agree_game.append(same); self.current_agree.append(same)
                self.db.add_position(game_id, board.ply(), board.fen(), best.uci(), mine.uci(), cp_b,
                                     math.tanh(cp_b / cfg.loss.value_scale), identify_game(board)["phase"], same,
                                     cp_after=cp_a, delta_cp=cp_a - cp_b, source="train")
                move = best if random.random() < beta else mine                          # teacher forcing
                self._learn()                                                            # <<< learn while playing
            else:
                move = opponent.play(board, chess.engine.Limit(time=cfg.opponent_time)).move
            board.push(move)

        # ----- game finished -----
        result = board.result(claim_draw=True) if board.is_game_over(claim_draw=True) else "1/2-1/2"
        white_pts = {"1-0": 1.0, "0-1": 0.0}.get(result, 0.5)
        score = white_pts if snake_color == chess.WHITE else 1.0 - white_pts
        gi = identify_game(board)
        pgn = chess.Board().variation_san(board.move_stack)
        self.db.finish_game(game_id, result, score, gi["termination"] or "MAX_PLIES", gi["opening"], board.ply(), pgn)
        self.games += 1
        self.agent.meta["games"] = self.games
        self.recent_scores.append(score)
        self.hist.append({"game": self.games, "score": score, "level": self.level,
                          "agree": float(np.mean(agree_game)) if agree_game else 0.0,
                          "delta": float(np.mean(delta_game)) if delta_game else 0.0, "result": result,
                          "snake": "white" if snake_color else "black", "opening": gi["opening"]})
        # curriculum: stronger opponent once Snake is holding its own
        if (len(self.recent_scores) == 10 and np.mean(self.recent_scores) >= cfg.promote_score
                and self.level < cfg.max_level):
            self.level += 1
            opponent.configure({"Skill Level": self.level})
            self.agent.meta["sf_level"] = self.level
            self.recent_scores.clear()
        if self.games % cfg.save_every == 0:
            self.agent.save(cfg.model_path)


def smooth(y, k):
    y = np.asarray(y, dtype=float)
    if len(y) < 2: return y
    k = max(1, min(k, len(y)))
    return np.convolve(y, np.ones(k) / k, mode="valid")


def show_dashboard(session):
    fig, ax = plt.subplots(1, 4, figsize=(21, 3.6))
    h = list(session.trainer.history)
    if len(h) > 5:
        s_, im, vl, pu = zip(*h)
        k = min(50, len(h)); xs = s_[k - 1:]
        ax[0].plot(xs, smooth(im, k), label="imitate (best move)")
        ax[0].plot(xs, smooth(vl, k), label="value (eval)")
        ax[0].plot(xs, smooth(pu, k), label="punishment")
        ax[0].legend(fontsize=8)
    ax[0].set_title("Training loss (live)"); ax[0].set_xlabel("gradient steps")
    g = session.hist
    gx = [x["game"] for x in g]
    if g:
        ax[1].plot(gx, [x["agree"] for x in g], alpha=.35)
        if len(g) >= 5: ax[1].plot(gx[4:], smooth([x["agree"] for x in g], 5), lw=2)
    ax[1].set_ylim(0, 1); ax[1].set_title("Agreement with Stockfish's best move"); ax[1].set_xlabel("game")
    if g:
        ax[2].plot(gx, [x["delta"] for x in g], alpha=.35, color="tab:red")
        if len(g) >= 5: ax[2].plot(gx[4:], smooth([x["delta"] for x in g], 5), lw=2, color="tab:red")
        ax[2].axhline(0, color="gray", lw=.8)
    ax[2].set_title("Avg eval change per Snake move (cp) -> 0 = no dumb moves"); ax[2].set_xlabel("game")
    if g:
        ax[3].step(gx, [x["level"] for x in g], where="post", label="Stockfish level")
        ax2 = ax[3].twinx()
        if len(g) >= 5:
            ax2.plot(gx[4:], smooth([x["score"] for x in g], 5), color="tab:orange", label="Snake score (5-game avg)")
        ax2.set_ylim(0, 1)
    ax[3].set_title("Opponent level & Snake's score"); ax[3].set_xlabel("game")
    plt.tight_layout(); plt.show()
    last = session.board.peek() if session.board.move_stack else None
    display(SVG(chess.svg.board(session.board, lastmove=last, size=320)))
    rate = np.mean(session.current_agree) if session.current_agree else 0
    print(f"status={session.status} | games={session.games} | opponent level={session.level} | "
          f"positions in DB={session.db.count()} | steps={session.agent.meta['train_steps']} | "
          f"recent agreement={rate:.1%}")
    if session.hist:
        print("last game:", session.hist[-1])
    if session.error:
        print(session.error)


def monitor(session, seconds=600, every=4):
    # Live view. Interrupting this cell (Stop button) does NOT stop training - it runs in a background thread.
    t0 = time.time()
    try:
        while time.time() - t0 < seconds and (session.running or time.time() - t0 < 3):
            clear_output(wait=True); show_dashboard(session); time.sleep(every)
        clear_output(wait=True); show_dashboard(session)
    except KeyboardInterrupt:
        print("Monitor stopped - training continues in the background.")

## Part 5 — 🚀 Train Snake in real time

* The first run starts from random weights. If `snake_model.pt` exists, training **resumes** from it (and the database keeps growing).
* Training runs in a **background thread**, so you can keep using the notebook — and the website in Part 6 plays the model *while it is still improving*.
* `session.stop()` stops cleanly and saves.

In [ ]:
cfg = Cfg()

# --- choose the brain (only matters when a NEW model is created; delete snake_model.pt to start over) ---
# cfg.arch, cfg.channels, cfg.blocks, cfg.heads = "transformer", 96, 4, 4

# --- tune how dumb moves are punished (can also be changed WHILE training runs) ---
# cfg.loss.punish_mode = "unlikelihood"   # or "pg" / "off"
# cfg.loss.w_punish    = 1.0
# cfg.loss.threshold_cp, cfg.loss.norm_cp = 40, 300
# cfg.loss.squash      = False

if os.path.exists(cfg.model_path):
    agent = SnakeAgent.load(cfg.model_path)
    print("Resumed Snake from", cfg.model_path, agent.meta)
else:
    agent = SnakeAgent(channels=cfg.channels, blocks=cfg.blocks, arch=cfg.arch, heads=cfg.heads)
    print("Created a fresh", cfg.arch, "Snake on", agent.device)

session = TrainingSession(agent, db, cfg)
session.start(n_games=200)         # <- change the number of games here

In [ ]:
monitor(session, seconds=900, every=4)     # live board + loss curves; re-run any time

In [ ]:
# Look inside the database any time
print("positions:", db.count(), "| games:", db.count("games"))
for row in db.query("SELECT id, snake_color, sf_level, result, termination, opening, n_plies FROM games ORDER BY id DESC LIMIT 8"):
    print(row)
print()
print("Average eval change per Snake move (cp), by block of 20 games - should rise towards 0:")
for row in db.query("SELECT (game_id-1)/20, COUNT(*), ROUND(AVG(delta_cp),1) FROM positions WHERE delta_cp IS NOT NULL AND source='train' GROUP BY 1"):
    print("  games %d-%d  positions=%d  avg delta=%s" % (row[0]*20+1, row[0]*20+20, row[1], row[2]))
print()
print("Agreement with Stockfish by game phase:")
for row in db.query("SELECT phase, COUNT(*), ROUND(AVG(agreed),3) FROM positions GROUP BY phase"):
    print(row)

In [ ]:
# session.stop()     # <- uncomment to stop & save (also saves automatically every few games)

### Optional: benchmark (no learning, no teacher forcing)
Plays Snake (with lookahead) against Stockfish at a fixed Skill Level and reports its score. Run it after a decent amount of training.

In [ ]:
def arena(agent, level=0, games=4, movetime=0.05, lookahead=True, max_plies=200, show_last=True):
    eng = chess.engine.SimpleEngine.popen_uci(SF_PATH)
    eng.configure({"Skill Level": level, "Threads": 1})
    pts, last = 0.0, None
    try:
        for g in range(games):
            snake_white = g % 2 == 0
            b = chess.Board()
            while not b.is_game_over(claim_draw=True) and b.ply() < max_plies:
                if (b.turn == chess.WHITE) == snake_white:
                    b.push(agent.select_move(b, lookahead=lookahead))
                else:
                    b.push(eng.play(b, chess.engine.Limit(time=movetime)).move)
            r = b.result(claim_draw=True) if b.is_game_over(claim_draw=True) else "1/2-1/2"
            pts += {"1-0": 1.0, "0-1": 0.0}.get(r, .5) if snake_white else {"1-0": 0.0, "0-1": 1.0}.get(r, .5)
            last = b
    finally:
        eng.quit()
    print(f"Snake vs Stockfish level {level}: {pts}/{games}")
    if show_last and last: display(SVG(chess.svg.board(last, size=300)))
    return pts / games

# arena(agent, level=0, games=4)

## Part 6 — The website on `localhost`

The next cells write `snake_ui.html` (the page) and `snake_web.py` (Flask API), then serve them from inside the notebook using the **same live `agent`** that is training. Open <http://127.0.0.1:5000>.

**What's on the page**
* **Snake's mind** — the probability (in %) Snake gives every legal move, how well it thinks the side to move stands, and its top 3 moves drawn as **arrows** on the board. Compared live with Stockfish's own win-%.
* **Modes** — *You vs Snake*, *You vs Stockfish* (level 0–20), **Watch: Snake vs Stockfish** (pause / step / speed slider).
* **Commentary & reactions** — every move is graded against Stockfish (best / good / inaccuracy / mistake / blunder), shows the centipawn loss and **how likely Snake thought that move was**, and both players react with faces and comments.
* **Board lab** — 8 colour themes + colour pickers, board size, coordinates, and a **board editor**: place any pieces, choose who moves, validate it, then let Snake and Stockfish fight from *your* position.
* **Learning lab** — when a game ends, **Snake studies it**: Stockfish labels the game, the positions go into the database, and Snake re-reads them for several epochs mixed with old memories, then saves. Every loss weight, threshold and study-session setting is a slider you can change live.

In [ ]:
%%writefile snake_ui.html
<!doctype html>
<html lang="en"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Snake - AI Chess Bot</title>
<style>
  :root{--light:#f0d9b5;--dark:#b58863;--wp:#ffffff;--bp:#111111;--bsize:520px;
        --bg:#1e1e24;--card:#2a2a33;--text:#ececf1;--accent:#6ee7a8;--muted:#9a9aa8}
  *{box-sizing:border-box} body{margin:0;background:var(--bg);color:var(--text);font-family:system-ui,Segoe UI,Roboto,sans-serif;
  display:flex;flex-direction:column;align-items:center;padding:14px}
  h1{margin:6px 0 0;font-size:1.6rem} h1 span{color:var(--accent)} .sub{opacity:.7;margin-bottom:12px;font-size:.9rem}
  #wrap{display:flex;flex-wrap:wrap;gap:18px;justify-content:center;width:100%;max-width:1100px;align-items:flex-start}
  #left{display:flex;flex-direction:column;gap:10px;align-items:center}
  #boardwrap{position:relative;width:min(92vw,var(--bsize));height:min(92vw,var(--bsize))}
  #board{display:grid;grid-template-columns:repeat(8,minmax(0,1fr));grid-template-rows:repeat(8,minmax(0,1fr));
         width:100%;height:100%;border:4px solid #444;border-radius:6px;overflow:hidden}
  #arrows{position:absolute;inset:4px;width:calc(100% - 8px);height:calc(100% - 8px);pointer-events:none}
  .sq{display:flex;align-items:center;justify-content:center;position:relative;cursor:pointer;user-select:none;
      min-width:0;min-height:0;overflow:hidden;font-size:calc(min(92vw,var(--bsize)) * .095);line-height:1}
  .sq.l{background:var(--light)} .sq.d{background:var(--dark)}
  .sq.sel{outline:4px solid #4da3ff;outline-offset:-4px} .sq.last{box-shadow:inset 0 0 0 100px rgba(255,235,59,.38)}
  .sq.chk{box-shadow:inset 0 0 0 100px rgba(255,60,60,.5)} .sq.hint{box-shadow:inset 0 0 0 100px rgba(80,200,120,.45)}
  .sq.tgt::after{content:"";width:28%;height:28%;border-radius:50%;background:rgba(0,0,0,.28);position:absolute}
  .sq.tgt.cap::after{width:88%;height:88%;background:none;border:5px solid rgba(0,0,0,.3)}
  .w{color:var(--wp);text-shadow:0 0 2px #000,0 0 2px #000,0 0 3px #000} .b{color:var(--bp);text-shadow:0 0 1px #fff,0 0 2px #fff}
  .coord{position:absolute;font-size:10px;opacity:.7;color:#000;mix-blend-mode:normal} .coord.r{top:2px;left:3px} .coord.f{bottom:1px;right:3px}
  body.nocoords .coord{display:none}
  #panel{width:min(92vw,360px);display:flex;flex-direction:column;gap:8px}
  .card{background:var(--card);border-radius:10px;padding:12px}
  #status{font-weight:600;min-height:2.4em}
  .row{display:flex;gap:8px;flex-wrap:wrap;align-items:center;margin-top:6px}
  button{background:var(--accent);color:#102;border:0;border-radius:8px;padding:8px 11px;font-weight:700;cursor:pointer}
  button.alt{background:#44445a;color:var(--text)} button:disabled{opacity:.5;cursor:wait} button.sm{padding:5px 8px;font-size:.8rem}
  select,input[type=color]{background:#44445a;color:var(--text);border:0;border-radius:6px;padding:4px}
  input[type=color]{width:34px;height:26px;padding:0;cursor:pointer} input[type=range]{accent-color:var(--accent);flex:1;min-width:80px}
  details{background:var(--card);border-radius:10px;padding:8px 12px} summary{cursor:pointer;font-weight:700;padding:2px 0}
  details[open] summary{margin-bottom:8px}
  .tag{font-size:.8rem;opacity:.8} label{font-size:.85rem;display:flex;gap:6px;align-items:center;margin-top:5px}
  .bar{height:14px;background:#111;border-radius:8px;overflow:hidden;border:1px solid #555;margin:5px 0}
  .fill{height:100%;transition:width .3s} #winfill{background:#6ee7a8} #barfill{background:#f2f2f2;width:50%}
  .mv{display:flex;align-items:center;gap:6px;font-family:ui-monospace,Menlo,monospace;font-size:.82rem;margin:2px 0}
  .mv .n{width:54px} .mv .pb{flex:1;height:10px;background:#1b1b22;border-radius:5px;overflow:hidden}
  .mv .pb i{display:block;height:100%;background:#4da3ff} .mv .p{width:52px;text-align:right}
  #log,#moves{font-family:ui-monospace,Menlo,monospace;font-size:.78rem;max-height:190px;overflow:auto;background:#222229;border-radius:6px;padding:6px;min-height:40px}
  .cm{padding:3px 4px;border-left:3px solid #666;margin:2px 0} .cm.best{border-color:#6ee7a8} .cm.good{border-color:#9be38a}
  .cm.inaccuracy{border-color:#f2d45c} .cm.mistake{border-color:#f29a4a} .cm.blunder{border-color:#ff5a5a;background:#3a2226}
  .cm .who{opacity:.75} .cm .d{opacity:.7} .cm .conf{color:#8fc3ff}
  .mood{display:flex;gap:10px;width:100%;justify-content:space-between}
  .mood div{flex:1;background:var(--card);border-radius:10px;padding:6px 10px;font-size:.85rem;min-height:44px}
  .mood b{display:block;font-size:.72rem;opacity:.7;font-weight:600}
  #palette{display:none;gap:4px;flex-wrap:wrap;justify-content:center;background:var(--card);border-radius:10px;padding:8px;max-width:var(--bsize)}
  #palette button{width:38px;height:38px;padding:0;font-size:1.5rem;background:#44445a;color:#fff} #palette button.on{outline:3px solid var(--accent)}
  #palette button.wp{color:var(--wp);text-shadow:0 0 2px #000,0 0 3px #000} #palette button.bp{color:var(--bp);text-shadow:0 0 2px #fff}
  .ctl{margin:6px 0} .ctl .h{font-size:.72rem;opacity:.6} .ctl .top{display:flex;justify-content:space-between;font-size:.82rem}
  .gh{font-size:.72rem;text-transform:uppercase;letter-spacing:.06em;opacity:.55;margin:10px 0 2px}
  .msg{font-size:.8rem;color:#ffb4b4;min-height:1em}
</style></head><body>
<h1>&#128013; <span>Snake</span> - AI Chess Bot</h1>
<div class="sub">A neural network that keeps learning - from Stockfish and from every game you play</div>
<div id="wrap">
  <div id="left">
    <div id="boardwrap"><div id="board"></div>
      <svg id="arrows" viewBox="0 0 8 8"></svg></div>
    <div id="palette"></div>
    <div class="mood" id="moods">
      <div><b>&#128013; SNAKE</b><span id="moodSnake">&mdash;</span></div>
      <div id="moodSFbox"><b id="moodSFname">STOCKFISH</b><span id="moodSF">&mdash;</span></div>
    </div>
  </div>
  <div id="panel">
    <div class="card">
      <div id="status">Loading...</div>
      <div class="tag" id="info"></div>
      <div id="sfbox" style="display:none">
        <label>Mode:&nbsp;<select id="mode" onchange="startNew()">
          <option value="human">You vs Snake</option>
          <option value="humansf">You vs Stockfish</option>
          <option value="spectate">Watch: Snake vs Stockfish</option></select></label>
        <label>Stockfish level: <b id="lvlv">3</b>
          <input type="range" id="lvl" min="0" max="20" value="3" oninput="$('lvlv').textContent=this.value"></label>
        <label id="snakewhiteRow"><input type="checkbox" id="snakewhite" checked> Snake plays White (watch mode)</label>
      </div>
      <div class="row" id="playbtns">
        <button onclick="newGame('white')">Play White</button>
        <button onclick="newGame('black')" class="alt">Play Black</button>
      </div>
      <div class="row" id="watchbtns" style="display:none">
        <button onclick="startWatch()">Start match</button>
        <button id="pausebtn" onclick="togglePause()" class="alt">Pause</button>
        <button onclick="stepOne()" class="alt">Step</button>
      </div>
      <label id="speedRow" style="display:none">Speed: <input type="range" id="speed" min="0" max="2500" value="500"></label>
      <label><input type="checkbox" id="look" checked> Smart mode (mate-in-1 + 1-ply lookahead)</label>
      <div class="tag" id="stats" style="margin-top:6px"></div>
    </div>

    <details open id="mindbox"><summary>&#129504; Snake's mind (prediction)</summary>
      <div class="tag" id="mindtitle">-</div>
      <div class="bar"><div class="fill" id="winfill" style="width:50%"></div></div>
      <div class="tag" id="wintext"></div>
      <div id="mindmoves"></div>
      <label><input type="checkbox" id="arrowsOn" checked onchange="drawArrows()"> Draw Snake's top moves as arrows</label>
    </details>

    <details open id="sfdetails" style="display:none"><summary>&#9889; Stockfish eval &amp; commentary</summary>
      <label><input type="checkbox" id="evalon" onchange="refreshAll()"> Evaluation bar</label>
      <div class="bar"><div class="fill" id="barfill"></div></div><div class="tag" id="evaltxt"></div>
      <label><input type="checkbox" id="hinton" onchange="refreshAll()"> Show Stockfish's best-move hint (green)</label>
      <label><input type="checkbox" id="commenton" checked onchange="refreshAll()"> Live commentary (how good was each move?)</label>
      <div id="log"><i>Commentary appears here.</i></div>
    </details>

    <details id="lab"><summary>&#127912; Board lab (customise)</summary>
      <div class="row"><label style="margin:0">Theme:&nbsp;<select id="preset" onchange="applyPreset(this.value)"></select></label>
        <button class="sm alt" onclick="resetTheme()">Reset</button></div>
      <div class="row">
        <label style="margin:0">Light <input type="color" id="cLight" oninput="themeChanged()"></label>
        <label style="margin:0">Dark <input type="color" id="cDark" oninput="themeChanged()"></label>
        <label style="margin:0">White pcs <input type="color" id="cWp" oninput="themeChanged()"></label>
        <label style="margin:0">Black pcs <input type="color" id="cBp" oninput="themeChanged()"></label>
      </div>
      <label>Size <input type="range" id="bsize" min="260" max="760" value="520" oninput="themeChanged()"></label>
      <label><input type="checkbox" id="coordsOn" checked onchange="themeChanged()"> Show coordinates</label>
      <div class="gh">Set up your own position</div>
      <div class="row"><button class="sm" onclick="toggleEditor()" id="edbtn">Edit position</button>
        <button class="sm alt" onclick="standardStart()">Standard start</button></div>
      <div id="edtools" style="display:none">
        <div class="row">
          <label style="margin:0">Side to move <select id="edTurn"><option value="w">White</option><option value="b">Black</option></select></label>
        </div>
        <div class="row"><label style="margin:0"><input type="checkbox" id="edK" checked>K</label>
          <label style="margin:0"><input type="checkbox" id="edQ" checked>Q</label>
          <label style="margin:0"><input type="checkbox" id="edk" checked>k</label>
          <label style="margin:0"><input type="checkbox" id="edq" checked>q</label>
          <span class="tag">castling</span></div>
        <div class="row"><button class="sm alt" onclick="edClear()">Clear board</button>
          <button class="sm alt" onclick="edReset()">Reset pieces</button>
          <button class="sm" onclick="edUse()">Use this position</button></div>
        <div class="tag">Pick a piece, click squares to place it (right-click or eraser removes). Then choose a mode and start.</div>
        <div class="msg" id="edmsg"></div>
      </div>
    </details>

    <details id="learnbox" style="display:none"><summary>&#128218; Learning lab (Snake studies its games)</summary>
      <div class="tag" id="learnmsg">Snake is resting.</div>
      <div class="bar"><div class="fill" id="learnfill" style="width:0;background:#4da3ff"></div></div>
      <div class="tag" id="learnres"></div>
      <div class="row"><button class="sm" id="studybtn" onclick="studyNow(true)">Study this game now</button></div>
      <div id="learnctl"></div>
    </details>

    <details open><summary>Moves</summary><div id="moves"></div></details>
  </div>
</div>
<script>
const $=id=>document.getElementById(id), sleep=ms=>new Promise(r=>setTimeout(r,ms));
const GLYPH={k:"\u265A\uFE0E",q:"\u265B\uFE0E",r:"\u265C\uFE0E",b:"\u265D\uFE0E",n:"\u265E\uFE0E",p:"\u265F\uFE0E"};
const STD_FEN="rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1";
let moves=[], startFen=null, st=null, me="white", sel=null, busy=false, watching=false, paused=false, stepFlag=false;
let caps={stockfish:false,learning:false}, evals={}, minds={}, annotated=new Set(), hintSq=null;
let gameId=0, runId=0, studiedKey=null, learnCfg=[], learnTimer=null;
let editor={on:false,piece:"P",grid:{}};
const mode=()=>caps.stockfish?$("mode").value:"human", lvl=()=>+$("lvl").value;

/* ---------------- themes ---------------- */
const PRESETS={
  "Classic wood":{light:"#f0d9b5",dark:"#b58863",wp:"#ffffff",bp:"#111111"},
  "Ocean":{light:"#cfe8f3",dark:"#4a8db7",wp:"#ffffff",bp:"#0b2239"},
  "Neon night":{light:"#2b2b5a",dark:"#14142e",wp:"#00ffd5",bp:"#ff3df2"},
  "Matrix":{light:"#0f3a14",dark:"#06200a",wp:"#9dff6b",bp:"#2fbf4a"},
  "Bubblegum":{light:"#ffd1e8",dark:"#ff8fc7",wp:"#ffffff",bp:"#5b2a86"},
  "Ice":{light:"#e6f2ff",dark:"#9db9d6",wp:"#ffffff",bp:"#233a52"},
  "Sunset":{light:"#ffe0b3",dark:"#e0704a",wp:"#fff7e6",bp:"#4a1c1c"},
  "Mono":{light:"#e8e8e8",dark:"#8a8a8a",wp:"#ffffff",bp:"#000000"}};
function applyColors(t){const s=document.documentElement.style;s.setProperty("--light",t.light);s.setProperty("--dark",t.dark);
  s.setProperty("--wp",t.wp);s.setProperty("--bp",t.bp);}
function readTheme(){return {light:$("cLight").value,dark:$("cDark").value,wp:$("cWp").value,bp:$("cBp").value,
  size:+$("bsize").value,coords:$("coordsOn").checked};}
function writeTheme(t){$("cLight").value=t.light;$("cDark").value=t.dark;$("cWp").value=t.wp;$("cBp").value=t.bp;
  if(t.size)$("bsize").value=t.size; if(t.coords!==undefined)$("coordsOn").checked=t.coords;}
function themeChanged(){const t=readTheme();applyColors(t);document.documentElement.style.setProperty("--bsize",t.size+"px");
  document.body.classList.toggle("nocoords",!t.coords);try{localStorage.setItem("snakeTheme",JSON.stringify(t))}catch(e){}}
function applyPreset(n){if(PRESETS[n]){writeTheme(PRESETS[n]);themeChanged();}}
function resetTheme(){writeTheme(Object.assign({size:520,coords:true},PRESETS["Classic wood"]));$("preset").value="Classic wood";themeChanged();}
function initTheme(){
  const p=$("preset"); Object.keys(PRESETS).forEach(n=>{const o=document.createElement("option");o.textContent=n;p.appendChild(o)});
  let t=null; try{t=JSON.parse(localStorage.getItem("snakeTheme")||"null")}catch(e){}
  writeTheme(t||Object.assign({size:520,coords:true},PRESETS["Classic wood"])); themeChanged();
}

/* ---------------- helpers ---------------- */
async function api(path, body){
  const r=await fetch(path,{method:body?"POST":"GET",headers:{"Content-Type":"application/json"},body:body?JSON.stringify(body):undefined});
  const j=await r.json(); if(!r.ok) throw new Error(j.error||"error"); return j;
}
const call=(path,extra)=>api(path,Object.assign({moves,start_fen:startFen},extra||{}));
function parseFen(fen){
  const g={}; fen.split(" ")[0].split("/").forEach((row,i)=>{let f=0;for(const ch of row){if(/\d/.test(ch)){f+=+ch}else{g["abcdefgh"[f]+(8-i)]=ch;f++}}});
  return g;
}
function sqXY(s){const f="abcdefgh".indexOf(s[0]), r=+s[1];
  return {x:(me==="white"?f:7-f)+.5, y:(me==="white"?8-r:r-1)+.5};}
const clip=x=>Math.max(-1000,Math.min(1000,x));
const winPctFromCp=cp=>50+50*Math.tanh(cp/400);
const fmtCp=e=>e.mate!==null&&e.mate!==undefined?`mate in ${Math.abs(e.mate)} for ${e.mate>0?"White":"Black"}`:((e.cp>=0?"+":"")+(e.cp/100).toFixed(2));
const why=t=>(t||"").toLowerCase().replace(/_/g," ");

/* ---------------- board rendering ---------------- */
function render(){
  if(!st&&!editor.on) return;
  const g=editor.on?editor.grid:parseFen(st.fen), last=(!editor.on&&moves.length)?moves[moves.length-1]:null, el=$("board");
  el.innerHTML="";
  const files=me==="white"?"abcdefgh":"hgfedcba", ranks=me==="white"?[8,7,6,5,4,3,2,1]:[1,2,3,4,5,6,7,8];
  const targets=(!editor.on&&sel&&st.legal[sel])?st.legal[sel]:[];
  ranks.forEach((r,ri)=>files.split("").forEach((f,fi)=>{
    const s=f+r, d=document.createElement("div"), light=("abcdefgh".indexOf(f)+r)%2===0;   // a1 is DARK
    d.className="sq "+(light?"l":"d"); d.dataset.sq=s;
    if(!editor.on){
      if(s===sel)d.classList.add("sel");
      if(last&&(s===last.slice(0,2)||s===last.slice(2,4)))d.classList.add("last");
      if(hintSq&&(s===hintSq[0]||s===hintSq[1]))d.classList.add("hint");
      if(targets.includes(s)){d.classList.add("tgt");if(g[s])d.classList.add("cap")}
    }
    const p=g[s];
    if(p){ if(!editor.on&&p.toLowerCase()==="k"&&st.in_check&&((p===p.toUpperCase())===(st.turn==="white")))d.classList.add("chk");
      d.innerHTML=`<span class="${p===p.toUpperCase()?"w":"b"}">${GLYPH[p.toLowerCase()]}</span>`;}
    if(fi===0){const c=document.createElement("span");c.className="coord r";c.textContent=r;d.appendChild(c)}
    if(ri===7){const c=document.createElement("span");c.className="coord f";c.textContent=f;d.appendChild(c)}
    d.onclick=()=>clickSq(s); d.oncontextmenu=e=>{e.preventDefault(); if(editor.on){delete editor.grid[s];render();}};
    el.appendChild(d);
  }));
  if(editor.on){ $("status").textContent="Editing position - place pieces, then press 'Use this position'.";
    $("info").textContent=""; drawArrows(); return; }
  const m=mode(), opp=m==="humansf"?"Stockfish":"Snake";
  let txt;
  if(m==="spectate"){
    const sw=$("snakewhite").checked, nm=c=>(c==="white")===sw?"Snake":`Stockfish L${lvl()}`;
    if(st.over) txt=st.result==="1/2-1/2"?`Draw (${why(st.termination)})`:`${nm(st.winner)} (${st.winner}) wins by ${why(st.termination)}`;
    else txt=paused?"Paused":`${nm(st.turn)} (${st.turn}) is thinking...`;
  } else if(st.over){
    txt=st.result==="1/2-1/2"?`Draw (${why(st.termination)})`:
      `${st.winner==="white"?"White":"Black"} wins by ${why(st.termination)}`+((st.winner===me)?" - you win!":` - ${opp} wins`);
  } else txt=st.turn===me?"Your move"+(st.in_check?" (you are in check!)":""):`${opp} is thinking...`;
  $("status").textContent=txt;
  $("info").textContent=`Opening: ${st.info.opening} | Phase: ${st.info.phase} | Move ${Math.floor(st.info.ply/2)+1}`;
  const san=st.san||[]; let h="";
  for(let i=0;i<san.length;i+=2)h+=`${i/2+1}. ${san[i]} ${san[i+1]||""}<br>`;
  const mv=$("moves"); mv.innerHTML=h||"<i>No moves yet</i>"; mv.scrollTop=mv.scrollHeight;
  const spec=m==="spectate";
  $("playbtns").style.display=spec?"none":"flex"; $("watchbtns").style.display=spec?"flex":"none";
  $("snakewhiteRow").style.display=spec?"flex":"none"; $("speedRow").style.display=spec?"flex":"none";
  $("moodSFname").textContent=m==="human"?"YOU":("STOCKFISH");
  $("studybtn").disabled=moves.length<2;
  drawArrows();
}

/* ---------------- Snake's mind: percentages + arrows ---------------- */
async function refreshMind(){
  if(!st||st.over||editor.on){ if(st&&st.over){$("mindtitle").textContent="Game over.";$("mindmoves").innerHTML="";$("wintext").textContent="";} return; }
  const ply=moves.length, gid=gameId;
  try{ const m=await call("/api/predict"); if(gid!==gameId)return; minds[ply]=m; if(ply===moves.length)renderMind(); }catch(e){}
}
function renderMind(){
  const ply=moves.length, m=minds[ply]; if(!m||!st||st.over)return;
  $("mindtitle").textContent=`${m.turn==="white"?"White":"Black"} to move - Snake's view of ${m.moves.length>=8?"the top 8":"all"} moves:`;
  $("winfill").style.width=m.win_pct+"%";
  let t=`Snake expects ${m.turn} to score ${m.win_pct.toFixed(1)}%`;
  const e=evals[ply]; if(e){ const w=winPctFromCp(clip(e.cp)), side=m.turn==="white"?w:100-w; t+=` | Stockfish: ${side.toFixed(1)}%`; }
  $("wintext").textContent=t;
  const mx=Math.max(...m.moves.map(x=>x.pct),1);
  $("mindmoves").innerHTML=m.moves.map(x=>`<div class="mv"><span class="n">${x.san}</span><span class="pb"><i style="width:${(x.pct/mx*100).toFixed(0)}%"></i></span><span class="p">${x.pct.toFixed(1)}%</span></div>`).join("");
  drawArrows();
}
function drawArrows(){
  const svg=$("arrows"); svg.innerHTML="";
  if(!$("arrowsOn").checked||editor.on||!st||st.over) return;
  const m=minds[moves.length]; if(!m) return;
  const cols=["#38e08c","#4da3ff","#ffb347"];
  let defs="<defs>"+cols.map((c,i)=>`<marker id="ah${i}" viewBox="0 0 10 10" refX="5" refY="5" markerWidth="3.2" markerHeight="3.2" orient="auto"><path d="M0,0 L10,5 L0,10 z" fill="${c}"/></marker>`).join("")+"</defs>";
  let body="";
  m.moves.slice(0,3).forEach((x,i)=>{
    const a=sqXY(x.uci.slice(0,2)), b=sqXY(x.uci.slice(2,4)), dx=b.x-a.x, dy=b.y-a.y, len=Math.hypot(dx,dy)||1, sh=.3;
    body+=`<line x1="${a.x}" y1="${a.y}" x2="${b.x-dx/len*sh}" y2="${b.y-dy/len*sh}" stroke="${cols[i]}" stroke-width="${(.1+x.pct/100*.22).toFixed(3)}" stroke-linecap="round" opacity="${(.4+x.pct/100*.55).toFixed(2)}" marker-end="url(#ah${i})"/>`;
  });
  svg.innerHTML=defs+body;
}

/* ---------------- Stockfish eval, hints, commentary ---------------- */
const evalWanted=()=>caps.stockfish&&($("evalon").checked||$("commenton").checked||$("hinton").checked||mode()==="spectate");
async function ensureEval(n){
  if(!caps.stockfish) return null;
  if(evals[n]!==undefined) return evals[n];
  if(n===moves.length&&st&&st.over){ evals[n]={cp:st.winner?(st.winner==="white"?10000:-10000):0,mate:null}; return evals[n]; }
  const gid=gameId;
  try{ const e=await api("/api/eval",{moves:moves.slice(0,n),start_fen:startFen}); if(gid!==gameId)return null; evals[n]=e; return e; }catch(err){ return null; }
}
async function ensureMind(n){
  if(minds[n]) return minds[n]; const gid=gameId;
  try{ const m=await api("/api/predict",{moves:moves.slice(0,n),start_fen:startFen}); if(gid!==gameId)return null; minds[n]=m; return m; }catch(e){ return null; }
}
async function refreshEvalBar(){
  hintSq=null; const n=moves.length, gid=gameId;
  if(!evalWanted()||!st){ $("evaltxt").textContent=""; $("barfill").style.width="50%"; render(); return; }
  const e=await ensureEval(n); if(!e||gid!==gameId||n!==moves.length) return;
  $("barfill").style.width=winPctFromCp(clip(e.cp))+"%";
  const humanTurn=mode()!=="spectate"&&!st.over&&st.turn===me;
  $("evaltxt").textContent=`Eval (White's view): ${fmtCp(e)}`+(humanTurn&&e.best_san&&$("hinton").checked?` | Stockfish suggests: ${e.best_san}`:"");
  hintSq=(humanTurn&&$("hinton").checked&&e.best)?[e.best.slice(0,2),e.best.slice(2,4)]:null;
  render(); renderMind();
}
function moverOf(n){ const opp=c=>c==="white"?"black":"white"; return ((moves.length-n)%2===0)?opp(st.turn):st.turn; }
function whoIs(color){
  const m=mode(); if(m==="spectate") return (color==="white")===$("snakewhite").checked?"Snake":"Stockfish";
  if(m==="humansf") return color===me?"You":"Stockfish";
  return color===me?"You":"Snake";
}
const MOVER_FACE={best:"&#128526;",good:"&#128578;",inaccuracy:"&#128517;",mistake:"&#128556;",blunder:"&#128165;&#128565;"};
const OTHER_FACE={best:"&#128558;",good:"&#129300;",inaccuracy:"&#128578;",mistake:"&#128527;",blunder:"&#129325;&#129395;"};
const SAYS={Snake:{mistake:["Hmm, ssso that was a slip.","Hsss... noted."],blunder:["Ouch. I felt that in my weights."]},
  Stockfish:{mistake:["I'll take that.","Interesting choice..."],blunder:["Thanks for the gift.","Is that all you've got?"]}};
function react(who,cls,other){
  const set=(name,html)=>{ if(name==="Snake")$("moodSnake").innerHTML=html; else if(name==="Stockfish")$("moodSF").innerHTML=html;
    else if(name==="You"&&mode()==="human")$("moodSF").innerHTML=html; };
  const lines=(SAYS[other]||{})[cls]; const say=lines?" "+lines[Math.floor(Math.random()*lines.length)]:"";
  set(who,MOVER_FACE[cls]+` <span class="tag">${cls}</span>`);
  set(other==="Stockfish"&&mode()==="human"?"You":other,OTHER_FACE[cls]+say);
}
async function annotateAll(){
  if(!caps.stockfish||!$("commenton").checked||!st) return;
  const gid=gameId;
  for(let n=1;n<=moves.length;n++){
    if(annotated.has(n)) continue;
    const b=await ensureEval(n-1), a=await ensureEval(n); if(gid!==gameId) return; if(!b||!a) continue;
    const mind=await ensureMind(n-1); if(gid!==gameId) return;
    if(annotated.has(n)) continue; annotated.add(n);
    const color=moverOf(n), loss=color==="white"?clip(b.cp)-clip(a.cp):clip(a.cp)-clip(b.cp), L=Math.max(0,loss);
    const cls=L<=10?"best":L<=50?"good":L<=100?"inaccuracy":L<=250?"mistake":"blunder";
    const who=whoIs(color), other=whoIs(color==="white"?"black":"white");
    const pct=mind&&mind.all[moves[n-1]]!==undefined?mind.all[moves[n-1]].toFixed(1)+"%":"-";
    const lg=$("log"); if(lg.querySelector("i")) lg.innerHTML="";
    const d=document.createElement("div"); d.className="cm "+cls;
    d.innerHTML=`<b>${Math.ceil(n/2)}${color==="white"?".":"..."} ${st.san[n-1]}</b> <span class="who">${who}</span> ${MOVER_FACE[cls]} ${cls} <span class="d">(${L>0?"-":"+"}${Math.abs(loss)}cp)</span> <span class="conf">Snake rated it ${pct}</span>`;
    lg.appendChild(d); lg.scrollTop=lg.scrollHeight;
    react(who,cls,other);
  }
}
function refreshAll(){ refreshMind(); refreshEvalBar().then(()=>annotateAll()); }

/* ---------------- game flow ---------------- */
function resetGame(){ gameId++; evals={}; minds={}; annotated=new Set(); hintSq=null; studiedKey=null; sel=null;
  $("log").innerHTML="<i>Commentary appears here.</i>"; $("moodSnake").innerHTML="&mdash;"; $("moodSF").innerHTML="&mdash;";
  $("mindmoves").innerHTML=""; $("arrows").innerHTML=""; }
async function clickSq(s){
  if(editor.on){ if(editor.piece==="x")delete editor.grid[s]; else editor.grid[s]=editor.piece; render(); return; }
  if(busy||!st||st.over||st.turn!==me||mode()==="spectate")return;
  const g=parseFen(st.fen);
  if(sel&&st.legal[sel]&&st.legal[sel].includes(s)){
    const uci=sel+s; sel=null; hintSq=null; busy=true;
    try{
      st=await call("/api/move",{move:uci,lookahead:$("look").checked,opponent:mode()==="humansf"?"stockfish":"snake",level:lvl()});
      moves=st.moves;
    }catch(e){alert(e.message)}
    busy=false; render(); refreshAll(); loadStats(); if(st.over)maybeStudy(); return;
  }
  sel=(g[s]&&st.legal[s])?s:null; render();
}
async function opponentMove(){
  const body={lookahead:$("look").checked,level:lvl()};
  return mode()==="humansf"?call("/api/sf_move",body):call("/api/snake",body);
}
async function newGame(color){
  runId++; watching=false; paused=false; resetGame(); moves=[]; me=color; busy=true;
  st=await call("/api/state"); moves=st.moves; render();
  if(st.turn!==me&&!st.over){ try{ st=await opponentMove(); moves=st.moves; }catch(e){alert(e.message)} }
  busy=false; render(); refreshAll();
}
function startNew(){ $("mode").value==="spectate"?startWatch():newGame("white"); }
function togglePause(){ paused=!paused; $("pausebtn").textContent=paused?"Resume":"Pause"; if(st)render(); }
function stepOne(){ paused=true; stepFlag=true; $("pausebtn").textContent="Resume"; }
async function startWatch(){
  const my=++runId; resetGame(); paused=false; $("pausebtn").textContent="Pause";
  moves=[]; const sw=$("snakewhite").checked; me=sw?"white":"black"; busy=true;
  st=await call("/api/state"); moves=st.moves; render(); watching=true; refreshAll();
  while(my===runId&&!st.over){
    if(paused&&!stepFlag){ await sleep(80); continue; }
    stepFlag=false;
    const snakeTurn=(st.turn==="white")===sw;
    try{
      await refreshMind();                                    // Snake's thoughts about THIS position, before the move
      if(my!==runId) return;
      st=await call(snakeTurn?"/api/snake":"/api/sf_move",{lookahead:$("look").checked,level:lvl()}); moves=st.moves;
    }catch(e){ alert(e.message); break; }
    if(my!==runId) return;
    render(); refreshAll(); loadStats();
    await sleep(+$("speed").value);
  }
  if(my===runId){ watching=false; busy=false; render(); if(st.over)maybeStudy(); }
}

/* ---------------- board editor ---------------- */
function buildPalette(){
  const p=$("palette"); p.innerHTML="";
  "KQRBNPkqrbnp".split("").forEach(ch=>{const b=document.createElement("button");
    b.className=(ch===ch.toUpperCase()?"wp":"bp"); b.innerHTML=GLYPH[ch.toLowerCase()]; b.title=ch; b.dataset.p=ch;
    b.onclick=()=>{editor.piece=ch;markPalette()}; p.appendChild(b)});
  const e=document.createElement("button"); e.textContent="\u2715"; e.title="eraser"; e.dataset.p="x"; e.onclick=()=>{editor.piece="x";markPalette()}; p.appendChild(e);
  markPalette();
}
function markPalette(){ document.querySelectorAll("#palette button").forEach(b=>b.classList.toggle("on",b.dataset.p===editor.piece)); }
function toggleEditor(){
  editor.on=!editor.on; runId++; watching=false;
  $("palette").style.display=editor.on?"flex":"none"; $("edtools").style.display=editor.on?"block":"none";
  $("edbtn").textContent=editor.on?"Close editor":"Edit position"; $("edmsg").textContent="";
  if(editor.on){ editor.grid=parseFen(st?st.fen:STD_FEN); me="white"; $("edTurn").value=(st&&st.turn==="black")?"b":"w"; }
  render();
}
function edClear(){ editor.grid={}; render(); }
function edReset(){ editor.grid=parseFen(STD_FEN); render(); }
function editorFen(){
  const rows=[]; for(let r=8;r>=1;r--){ let row="",e=0;
    for(const f of "abcdefgh"){ const p=editor.grid[f+r]; if(p){ if(e){row+=e;e=0} row+=p; } else e++; }
    if(e)row+=e; rows.push(row); }
  const c=(["K","Q","k","q"].filter(x=>$("ed"+x).checked)).join("")||"-";
  return rows.join("/")+" "+$("edTurn").value+" "+c+" - 0 1";
}
async function edUse(){
  try{
    const v=await api("/api/validate_fen",{fen:editorFen()});
    if(!v.ok){ $("edmsg").textContent="Not a legal position: "+v.problems.join(", "); return; }
    startFen=(v.fen===STD_FEN)?null:v.fen; toggleEditor(); startNew();
  }catch(e){ $("edmsg").textContent=e.message; }
}
function standardStart(){ startFen=null; if(editor.on)toggleEditor(); startNew(); }

/* ---------------- learning lab ---------------- */
const snakeColor=()=>mode()==="spectate"?($("snakewhite").checked?"white":"black"):mode()==="humansf"?"none":(me==="white"?"black":"white");
const autoStudy=()=>{const c=learnCfg.find(x=>x.name==="auto_study");return c?c.value:false};
async function maybeStudy(){
  if(!caps.learning||!st||!st.over) return;
  const key=(startFen||"")+"|"+moves.join(" "); if(studiedKey===key) return;
  if(autoStudy()){ studiedKey=key; studyNow(false); }
}
async function studyNow(manual){
  if(!caps.learning||moves.length<2) return;
  try{ const r=await call("/api/learn",{snake_color:snakeColor()});
    if(!r.queued){ $("learnmsg").textContent=r.reason||"Not queued."; return; }
    $("learnbox").open=true; pollLearn(); }catch(e){ $("learnmsg").textContent=e.message; }
}
function pollLearn(){ if(learnTimer)return; const tick=async()=>{
    try{ const s=await api("/api/learn_status"); $("learnmsg").textContent=s.message+(s.queue?` (+${s.queue} queued)`:"");
      $("learnfill").style.width=(s.total?Math.min(100,s.progress/s.total*100):0)+"%";
      if(s.error) $("learnres").textContent=s.error.split("\n").slice(-3).join(" ");
      else if(s.last){ const l=s.last; $("learnres").innerHTML=`Last study: ${l.positions} positions, ${l.punished} dumb moves punished, ${l.rewarded} rewarded, `
        +`avg eval change per Snake move ${l.avg_delta_cp===null?"-":l.avg_delta_cp+"cp"}, agreement with Stockfish ${l.agree_before}% &rarr; <b>${l.agree_after}%</b>, `
        +`${l.steps} gradient steps (total ${l.train_steps_total}).`; }
      if(s.state==="done"||s.state==="idle"||s.state==="error"){ clearInterval(learnTimer); learnTimer=null; loadStats(); }
    }catch(e){ clearInterval(learnTimer); learnTimer=null; } };
  learnTimer=setInterval(tick,900); tick(); }
async function buildLearn(){
  try{ learnCfg=await api("/api/learn_config"); }catch(e){ return; }
  const box=$("learnctl"); box.innerHTML=""; let lastGroup="";
  learnCfg.forEach(c=>{
    const grp=c.name.startsWith("loss.")?"Loss function":"Study session";
    if(grp!==lastGroup){ const h=document.createElement("div"); h.className="gh"; h.textContent=grp; box.appendChild(h); lastGroup=grp; }
    const d=document.createElement("div"); d.className="ctl"; let inp, valEl=null;
    const top=document.createElement("div"); top.className="top"; const lab=document.createElement("span"); lab.textContent=c.label; top.appendChild(lab);
    if(c.type==="bool"){ inp=document.createElement("input"); inp.type="checkbox"; inp.checked=c.value; top.appendChild(inp); d.appendChild(top); }
    else if(c.type==="select"){ inp=document.createElement("select"); c.options.forEach(o=>{const op=document.createElement("option");op.textContent=o;inp.appendChild(op)}); inp.value=c.value; top.appendChild(inp); d.appendChild(top); }
    else { valEl=document.createElement("b"); valEl.textContent=c.value; top.appendChild(valEl); d.appendChild(top);
      inp=document.createElement("input"); inp.type="range"; inp.min=c.min; inp.max=c.max; inp.step=c.step; inp.value=c.value; d.appendChild(inp);
      inp.oninput=()=>{valEl.textContent=inp.value}; }
    if(c.help){ const h=document.createElement("div"); h.className="h"; h.textContent=c.help; d.appendChild(h); }
    inp.name=c.name;
    inp.onchange=async()=>{ const v=c.type==="bool"?inp.checked:inp.value;
      try{ const r=await api("/api/learn_config",{name:c.name,value:v}); c.value=r.value; if(valEl)valEl.textContent=r.value; }catch(e){ alert(e.message); } };
    box.appendChild(d);
  });
}
async function loadStats(){
  try{const s=await api("/api/stats"); $("stats").textContent=
    `Snake (${s.arch}): ${s.train_steps} training steps | ${s.games} live-training games | ${s.web_games} website games studied | Stockfish level ${s.sf_level}.`}catch(e){}
}

/* ---------------- boot ---------------- */
(async()=>{
  initTheme(); buildPalette();
  try{ caps=await api("/api/capabilities"); }catch(e){}
  if(caps.stockfish){ $("sfbox").style.display="block"; $("sfdetails").style.display="block"; }
  if(caps.learning){ $("learnbox").style.display="block"; await buildLearn(); }
  if(!caps.stockfish){ $("moodSFbox").style.display="none"; }
  await newGame("white"); loadStats();
})();
</script></body></html>

In [ ]:
%%writefile snake_web.py
"""snake_web.py - Flask website + JSON API for Snake (used locally AND when deployed)."""
import atexit
import os
import threading
from pathlib import Path
import chess
import chess.engine
from flask import Flask, jsonify, request, Response
from snake_core import identify_game


class StockfishService:
    """Thread-safe wrapper around ONE Stockfish process (talked to over UCI). Opened lazily."""
    def __init__(self, path):
        self.path, self.lock, self.engine, self.name = path, threading.Lock(), None, "Stockfish"

    def _ensure(self):
        if self.engine is None:
            self.engine = chess.engine.SimpleEngine.popen_uci(self.path)
            self.engine.configure({"Threads": 1, "Hash": 64})
            self.name = self.engine.id.get("name", "Stockfish")
            # must run BEFORE python joins the engine's helper thread, or exit hangs
            getattr(threading, "_register_atexit", atexit.register)(self.close)

    def close(self):
        with self.lock:
            if self.engine is not None:
                try:
                    self.engine.quit()
                except Exception:
                    pass
                self.engine = None

    def play(self, board, level, seconds):
        with self.lock:
            self._ensure()
            self.engine.configure({"Skill Level": max(0, min(20, int(level)))})
            return self.engine.play(board, chess.engine.Limit(time=float(seconds))).move

    def analyse(self, board, seconds=0.2):
        with self.lock:
            self._ensure()
            self.engine.configure({"Skill Level": 20})
            info = self.engine.analyse(board, chess.engine.Limit(time=seconds))
        sc = info["score"].white()
        pv = info.get("pv") or []
        return {"cp": sc.score(mate_score=10000), "mate": sc.mate(),
                "best": pv[0].uci() if pv else None,
                "best_san": board.san(pv[0]) if pv else None}


UI_FILE = Path(__file__).with_name("snake_ui.html")


def _build_board(moves, start_fen=None):
    board = chess.Board(start_fen) if start_fen else chess.Board()
    for u in moves:
        board.push_uci(u)          # raises ValueError if illegal -> handled by caller
    return board


def _state(board):
    legal = {}
    for m in board.legal_moves:
        legal.setdefault(chess.square_name(m.from_square), []).append(chess.square_name(m.to_square))
    replay, san = board.root().copy(), []
    for m in board.move_stack:
        san.append(replay.san(m)); replay.push(m)
    info = identify_game(board)
    return {"fen": board.fen(), "moves": [m.uci() for m in board.move_stack], "san": san,
            "start_fen": board.root().fen() if board.root().fen() != chess.STARTING_FEN else None,
            "turn": "white" if board.turn else "black", "legal": legal, "in_check": board.is_check(),
            "over": info["over"], "result": info["result"], "termination": info["termination"],
            "winner": info["winner"], "info": info}


def create_app(agent, stats_fn=None, stockfish_path=None, hub=None):
    app = Flask(__name__)
    sf = StockfishService(stockfish_path) if stockfish_path and os.path.exists(stockfish_path) else None

    def bad(msg, code=400):
        return jsonify({"error": msg}), code

    def read_board():
        data = request.get_json(force=True, silent=True) or {}
        try:
            return _build_board(data.get("moves", []), data.get("start_fen")), data
        except (ValueError, TypeError):
            return None, data

    @app.get("/")
    def index():
        return Response(UI_FILE.read_text(encoding="utf-8"), mimetype="text/html")

    @app.get("/api/capabilities")
    def capabilities():
        return jsonify(stockfish=sf is not None, learning=bool(hub and hub.enabled),
                       arch=agent.meta.get("arch", "resnet"))

    @app.get("/health")
    def health():
        return jsonify(ok=True)

    @app.get("/api/stats")
    def stats():
        s = stats_fn() if stats_fn else dict(agent.meta)
        return jsonify({"train_steps": s.get("train_steps", agent.meta.get("train_steps", 0)),
                        "games": s.get("games", agent.meta.get("games", 0)),
                        "web_games": agent.meta.get("web_games", 0),
                        "sf_level": s.get("sf_level", agent.meta.get("sf_level", 0)),
                        "arch": agent.meta.get("arch", "resnet")})

    @app.post("/api/state")
    def state():
        board, _ = read_board()
        return bad("invalid move list or start position") if board is None else jsonify(_state(board))

    @app.post("/api/validate_fen")
    def validate_fen():
        """Check a position built in the board editor; normalises impossible castling rights."""
        fen = str((request.get_json(force=True, silent=True) or {}).get("fen", ""))
        try:
            b = chess.Board(fen)
        except ValueError as e:
            return jsonify(ok=False, problems=[str(e)])
        b.castling_rights = b.clean_castling_rights()
        st = b.status()
        if st != chess.STATUS_VALID:
            names = [f.name.lower().replace("_", " ") for f in chess.Status if f in st and f != chess.STATUS_VALID]
            return jsonify(ok=False, problems=names)
        if b.is_game_over():
            return jsonify(ok=False, problems=["the game is already over in this position"])
        return jsonify(ok=True, fen=b.fen())

    @app.post("/api/move")
    def move():
        board, data = read_board()
        if board is None:
            return bad("invalid move list")
        if board.is_game_over(claim_draw=True):
            return bad("game is over")
        uci = str(data.get("move", ""))
        try:
            m = chess.Move.from_uci(uci)
        except ValueError:
            return bad("malformed move")
        piece = board.piece_at(m.from_square)            # auto-queen
        if piece and piece.piece_type == chess.PAWN and chess.square_rank(m.to_square) in (0, 7) and not m.promotion:
            m.promotion = chess.QUEEN
        if m not in board.legal_moves:                   # LEGAL MOVE IDENTIFIER on the server side
            return bad(f"illegal move: {uci}")
        board.push(m)
        if not board.is_game_over(claim_draw=True):
            if data.get("opponent") == "stockfish" and sf:
                board.push(sf.play(board, data.get("level", 3), 0.3))
            else:
                board.push(agent.select_move(board, lookahead=bool(data.get("lookahead", True))))
        return jsonify(_state(board))

    @app.post("/api/snake")
    def snake():
        board, data = read_board()
        if board is None:
            return bad("invalid move list")
        if board.is_game_over(claim_draw=True):
            return bad("game is over")
        board.push(agent.select_move(board, lookahead=bool(data.get("lookahead", True))))
        return jsonify(_state(board))

    @app.post("/api/sf_move")
    def sf_move():
        if not sf:
            return bad("Stockfish is not available on this server", 404)
        board, data = read_board()
        if board is None:
            return bad("invalid move list")
        if board.is_game_over(claim_draw=True):
            return bad("game is over")
        board.push(sf.play(board, data.get("level", 3), 0.3))
        return jsonify(_state(board))

    @app.post("/api/eval")
    def evaluate():
        if not sf:
            return bad("Stockfish is not available on this server", 404)
        board, _ = read_board()
        if board is None or board.is_game_over(claim_draw=True):
            return bad("invalid or finished game")
        return jsonify(sf.analyse(board))

    @app.post("/api/predict")
    def predict():
        """Snake's own thoughts about the position: % per legal move + how well it thinks it stands."""
        board, _ = read_board()
        if board is None or board.is_game_over(claim_draw=True):
            return bad("invalid or finished game")
        return jsonify(agent.predict(board, top=8))

    # ---------------- learning lab ----------------
    @app.get("/api/learn_config")
    def learn_config_get():
        if not hub:
            return bad("learning is not enabled on this server", 404)
        from snake_learn import cfg_ui
        return jsonify(cfg_ui(hub.cfg))

    @app.post("/api/learn_config")
    def learn_config_set():
        if not hub:
            return bad("learning is not enabled on this server", 404)
        from snake_learn import cfg_set, cfg_get, UI_META
        data = request.get_json(force=True, silent=True) or {}
        name = data.get("name")
        if name not in UI_META:
            return bad("unknown setting")
        try:
            return jsonify(name=name, value=cfg_set(hub.cfg, name, data.get("value")))
        except (ValueError, TypeError) as e:
            return bad(str(e))

    @app.post("/api/learn")
    def learn():
        if not hub:
            return bad("learning is not enabled on this server", 404)
        data = request.get_json(force=True, silent=True) or {}
        try:
            board = _build_board(data.get("moves", [])[:600], data.get("start_fen"))
        except (ValueError, TypeError):
            return bad("invalid move list")
        return jsonify(hub.submit([m.uci() for m in board.move_stack], data.get("start_fen"),
                                  str(data.get("snake_color", "white")), "web"))

    @app.get("/api/learn_status")
    def learn_status():
        if not hub:
            return bad("learning is not enabled on this server", 404)
        return jsonify(hub.get_status())

    return app

In [ ]:
import importlib, snake_web; importlib.reload(snake_web)
from snake_web import create_app
from werkzeug.serving import make_server

class ServerThread(threading.Thread):
    def __init__(self, app, host="127.0.0.1", port=5000):
        super().__init__(daemon=True)
        self.server = make_server(host, port, app, threaded=True)
    def run(self): self.server.serve_forever()
    def shutdown(self): self.server.shutdown()

def live_stats():
    return {"train_steps": agent.meta["train_steps"], "games": agent.meta.get("games", 0), "sf_level": session.level}

# The hub shares the live trainer AND the same LossCfg, so the website's sliders also steer live training.
hub = LearningHub(agent, db, SF_PATH, cfg=LearnCfg(loss=cfg.loss), trainer=session.trainer, model_path=cfg.model_path)
app = create_app(agent, stats_fn=live_stats, stockfish_path=SF_PATH, hub=hub)
try:
    server.shutdown()          # restart if re-running this cell
except NameError:
    pass
server = ServerThread(app, port=5000)
server.start()
print("Snake is live at  http://127.0.0.1:5000   (stop with server.shutdown())")

## Part 7 — Put it on a real website

A public site only needs **Snake's weights + the Flask app** (no Stockfish, no database). This cell builds a `deploy/` folder with everything:

```
deploy/
  app.py  snake_core.py  snake_web.py  snake_ui.html  snake_model.pt
  requirements.txt  Dockerfile  README.md
```

**Option A — Hugging Face Spaces (free, easiest for ML bots)**
1. Create an account → *New Space* → SDK: **Docker** → blank.
2. Upload everything inside `deploy/` (drag-and-drop in the web UI, or `git push`). The included `README.md` already has the Space header (`sdk: docker`, `app_port: 7860`).
3. The Space builds and gives you a public URL like `https://<you>-snake.hf.space`.

**Option B — Render / Railway / Fly.io:** push `deploy/` to a GitHub repo → create a *Web Service* from the repo → it detects the `Dockerfile` and serves on `$PORT` automatically.

**Option C — your own VPS:** `docker build -t snake . && docker run -p 80:7860 snake`

> The public bundle has **no Stockfish and no learning hub**, so the Stockfish modes, eval bar and Learning lab are hidden there automatically (they appear on localhost). Snake's predictions, arrows, board editor and themes still work online. Learning on a public site is deliberately off: anyone could feed it bad games. To add them online you'd install Stockfish inside the Docker image and set `stockfish_path` in `app.py` — if you do, you are distributing a GPLv3 binary, so keep its licence and offer its source.
>
> Free-tier limits (RAM, sleeping apps) change over time, so check the provider's current docs. The model is ~35 MB and runs on CPU, so the smallest tier is enough for light use.
>
> Re-run this cell after more training to refresh `deploy/snake_model.pt`, then re-upload that one file.

In [ ]:
import shutil
from pathlib import Path

agent.save(cfg.model_path)                         # make sure the latest weights are on disk
d = Path("deploy"); d.mkdir(exist_ok=True)
for f in ["snake_core.py", "snake_web.py", "snake_ui.html"]:
    shutil.copy(f, d / f)
shutil.copy(cfg.model_path, d / "snake_model.pt")

(d / "app.py").write_text("""import os
from snake_core import SnakeAgent
from snake_web import create_app

agent = SnakeAgent.load(os.environ.get("SNAKE_MODEL", "snake_model.pt"), device="cpu")
app = create_app(agent)

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=int(os.environ.get("PORT", 7860)))
""")
(d / "requirements.txt").write_text("flask\ngunicorn\npython-chess\nnumpy\n")
(d / "Dockerfile").write_text("""FROM python:3.11-slim
WORKDIR /app
RUN pip install --no-cache-dir torch --index-url https://download.pytorch.org/whl/cpu
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY . .
ENV PORT=7860
EXPOSE 7860
CMD gunicorn -b 0.0.0.0:${PORT} -w 1 --threads 4 --timeout 120 app:app
""")
(d / "README.md").write_text("""---
title: Snake Chess Bot
emoji: 🐍
colorFrom: green
colorTo: gray
sdk: docker
app_port: 7860
---

# Snake - AI chess bot
Neural-network chess bot trained live against Stockfish (GPLv3, used as an external UCI engine).
This project uses python-chess (GPL-3.0+); distribute it under GPLv3-compatible terms and provide the source.
""")
print("Deploy bundle ready:", sorted(p.name for p in d.iterdir()))
print("Model size: %.1f MB" % (os.path.getsize(d / "snake_model.pt") / 1e6))

### Test the deploy bundle locally (optional)
```bash
cd deploy
pip install -r requirements.txt   # + torch
python app.py                     # then open http://127.0.0.1:7860
```

## Where to go next
* **Make Snake stronger:** more games / GPU; bigger net (`SnakeAgent(channels=128, blocks=8)`); `teacher_time=0.2`; add multi-PV soft targets; add a real tree search (MCTS) driven by the value head.
* **Learn from website visitors:** log human-vs-Snake positions into `positions`, label them with Stockfish offline, and fine-tune.
* **Self-play phase:** once Snake beats Stockfish level ~5, add Snake-vs-Snake games labelled by the value head + game result (AlphaZero-style).